In [1]:
import pandas as pd
import numpy as np

ratings = pd.read_csv("data/ratings.csv")
movies = pd.read_csv("data/movies.csv")

print("=== Original Data ===")
print("Ratings shape:", ratings.shape)
print("Number of users:", ratings["userId"].nunique())
print("Number of rated movies:", ratings["movieId"].nunique())

=== Original Data ===
Ratings shape: (100836, 4)
Number of users: 610
Number of rated movies: 9724


In [2]:
ratings_sorted = ratings.sort_values(
    by=["userId", "timestamp", "movieId"]
).copy()

# Assign rating order for each user
ratings_sorted["rating_order"] = (
    ratings_sorted.groupby("userId").cumcount()
)

# Count all ratings for each user
ratings_sorted["user_rating_count"] = (
    ratings_sorted.groupby("userId")["movieId"]
    .transform("count")
)

# Assign each user's most recent 20% of ratings to the test set
test_count = np.maximum(
    1,
    np.ceil(
        ratings_sorted["user_rating_count"] * 0.2
    ).astype(int)
)

train_mask = (
    ratings_sorted["rating_order"]
    < ratings_sorted["user_rating_count"] - test_count
)

train_ratings = ratings_sorted[
    train_mask
].copy()

test_ratings = ratings_sorted[
    ~train_mask
].copy()

print("=== Train-Test Split ===")
print("Train ratings:", len(train_ratings))
print("Test ratings:", len(test_ratings))

print("Train users:", train_ratings["userId"].nunique())
print("Test users:", test_ratings["userId"].nunique())

=== Train-Test Split ===
Train ratings: 80419
Test ratings: 20417
Train users: 610
Test users: 610


In [3]:
print("=== Split Validation ===")

print(
    "Total ratings:",
    len(train_ratings) + len(test_ratings)
)

print(
    "Original ratings:",
    len(ratings)
)

print(
    "Train-Test overlap:",
    len(
        pd.merge(
            train_ratings[["userId", "movieId"]],
            test_ratings[["userId", "movieId"]],
            on=["userId", "movieId"]
        )
    )
)

print(
    "Minimum train ratings per user:",
    train_ratings.groupby("userId").size().min()
)

print(
    "Minimum test ratings per user:",
    test_ratings.groupby("userId").size().min()
)

=== Split Validation ===
Total ratings: 100836
Original ratings: 100836
Train-Test overlap: 0
Minimum train ratings per user: 16
Minimum test ratings per user: 4


In [4]:
train_movie_stats = (
    train_ratings
    .groupby("movieId")["rating"]
    .agg(["count", "mean"])
    .reset_index()
)

train_movie_stats = train_movie_stats.rename(
    columns={
        "count": "rating_count",
        "mean": "rating_mean"
    }
)

print("=== Train Movie Statistics ===")
print(train_movie_stats.head(10))

print(
    "\nMovies rated in training data:",
    len(train_movie_stats)
)

=== Train Movie Statistics ===
   movieId  rating_count  rating_mean
0        1           201     3.915423
1        2            92     3.413043
2        3            41     3.280488
3        4             4     2.125000
4        5            39     3.025641
5        6            87     3.925287
6        7            43     3.058140
7        8             6     2.500000
8        9            12     3.166667
9       10           122     3.504098

Movies rated in training data: 8230


In [5]:
movies_train = movies.merge(
    train_movie_stats,
    on="movieId",
    how="left",
    validate="one_to_one"
)

global_mean_train = train_ratings["rating"].mean()

movies_train["rating_count"] = (
    movies_train["rating_count"].fillna(0)
)

movies_train["rating_mean"] = (
    movies_train["rating_mean"].fillna(global_mean_train)
)

print("=== Training Movie Data ===")
print(movies_train.head())

print("\nShape:", movies_train.shape)
print("Training global mean:", global_mean_train)

=== Training Movie Data ===
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  rating_count  rating_mean  
0  Adventure|Animation|Children|Comedy|Fantasy         201.0     3.915423  
1                   Adventure|Children|Fantasy          92.0     3.413043  
2                               Comedy|Romance          41.0     3.280488  
3                         Comedy|Drama|Romance           4.0     2.125000  
4                                       Comedy          39.0     3.025641  

Shape: (9742, 5)
Training global mean: 3.513013093920591


In [6]:
m = 10

v = movies_train["rating_count"]
R = movies_train["rating_mean"]
C = global_mean_train

movies_train["weighted_rating"] = (
    (v / (v + m)) * R
    + (m / (v + m)) * C
)

print("=== Training Weighted Ratings ===")

print(
    movies_train[
        [
            "title",
            "rating_count",
            "rating_mean",
            "weighted_rating"
        ]
    ].sort_values(
        by="weighted_rating",
        ascending=False
    ).head(10).to_string(index=False)
)

=== Training Weighted Ratings ===
                                                                      title  rating_count  rating_mean  weighted_rating
                                           Shawshank Redemption, The (1994)         281.0     4.416370         4.385327
                                                      Godfather, The (1972)         175.0     4.302857         4.260163
Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)          87.0     4.304598         4.222991
                                                          Fight Club (1999)         195.0     4.256410         4.220147
                                                       Departed, The (2006)          87.0     4.287356         4.207527
                                                  Lawrence of Arabia (1962)          38.0     4.381579         4.200628
                                                 Princess Bride, The (1987)         129.0     4.251938         4.198778
      

In [7]:
min_train_rating = movies_train["weighted_rating"].min()
max_train_rating = movies_train["weighted_rating"].max()

movies_train["normalized_rating"] = (
    (
        movies_train["weighted_rating"]
        - min_train_rating
    )
    / (max_train_rating - min_train_rating)
)

print("=== Training Normalization ===")

print("Minimum:", movies_train["normalized_rating"].min())
print("Maximum:", movies_train["normalized_rating"].max())

print(
    "Missing values:",
    movies_train["normalized_rating"].isnull().sum()
)

=== Training Normalization ===
Minimum: 0.0
Maximum: 1.0
Missing values: 0


In [8]:
previous_movies = pd.read_csv(
    "data/processed/movie_features.csv"
)

rating_comparison = movies_train[
    ["movieId", "title", "weighted_rating"]
].merge(
    previous_movies[
        ["movieId", "weighted_rating"]
    ],
    on="movieId",
    suffixes=("_train", "_full")
)

rating_comparison["difference"] = (
    rating_comparison["weighted_rating_train"]
    - rating_comparison["weighted_rating_full"]
)

print("=== Train vs Full Rating Comparison ===")

print(
    rating_comparison[
        [
            "title",
            "weighted_rating_train",
            "weighted_rating_full",
            "difference"
        ]
    ].head(10).to_string(index=False)
)

print(
    "\nMovies with changed weighted ratings:",
    (
        rating_comparison["difference"].abs() > 1e-10
    ).sum()
)

=== Train vs Full Rating Comparison ===
                             title  weighted_rating_train  weighted_rating_full  difference
                  Toy Story (1995)               3.896351              3.902291   -0.005940
                    Jumanji (1995)               3.422844              3.437630   -0.014785
           Grumpier Old Men (1995)               3.326081              3.298638    0.027443
          Waiting to Exhale (1995)               3.116438              3.030328    0.086110
Father of the Bride Part II (1995)               3.125105              3.144332   -0.019227
                       Heat (1995)               3.882785              3.906389   -0.023604
                    Sabrina (1995)               3.143965              3.234618   -0.090654
               Tom and Huck (1995)               3.133133              3.223087   -0.089954
               Sudden Death (1995)               3.324097              3.269830    0.054267
                  GoldenEye (1995)      

In [9]:
available_genres = [
    "Action", "Adventure", "Animation",
    "Children", "Comedy", "Crime",
    "Documentary", "Drama", "Fantasy",
    "Film-Noir", "Horror", "Musical",
    "Mystery", "Romance", "Sci-Fi",
    "Thriller", "War", "Western"
]

genre_matrix = movies_train["genres"].str.get_dummies(
    sep="|"
)

genre_matrix = genre_matrix[available_genres]

genre_matrix["movieId"] = movies_train["movieId"]

print("=== Genre Matrix ===")
print(genre_matrix.head())

=== Genre Matrix ===
   Action  Adventure  Animation  Children  Comedy  Crime  Documentary  Drama  \
0       0          1          1         1       1      0            0      0   
1       0          1          0         1       0      0            0      0   
2       0          0          0         0       1      0            0      0   
3       0          0          0         0       1      0            0      1   
4       0          0          0         0       1      0            0      0   

   Fantasy  Film-Noir  Horror  Musical  Mystery  Romance  Sci-Fi  Thriller  \
0        1          0       0        0        0        0       0         0   
1        1          0       0        0        0        0       0         0   
2        0          0       0        0        0        1       0         0   
3        0          0       0        0        0        1       0         0   
4        0          0       0        0        0        0       0         0   

   War  Western  movieId  
0 

In [10]:
movie_features_train = movies_train.merge(
    genre_matrix,
    on="movieId",
    how="left",
    validate="one_to_one"
)

print("=== Final Training Movie Features ===")

print("Shape:", movie_features_train.shape)

print(
    "Missing values:",
    movie_features_train.isnull().sum().sum()
)

print(
    "Duplicate movie IDs:",
    movie_features_train["movieId"].duplicated().sum()
)

print(
    "Movies with valid genres:",
    (
        movie_features_train[available_genres]
        .sum(axis=1) > 0
    ).sum()
)

=== Final Training Movie Features ===
Shape: (9742, 25)
Missing values: 0
Duplicate movie IDs: 0
Movies with valid genres: 9708


In [11]:
def precision_at_k(
    recommended_movie_ids,
    relevant_movie_ids,
    k=10
):
    recommended = recommended_movie_ids[:k]

    relevant = set(relevant_movie_ids)

    hits = sum(
        movie_id in relevant
        for movie_id in recommended
    )

    return hits / k

In [12]:
def recall_at_k(
    recommended_movie_ids,
    relevant_movie_ids,
    k=10
):
    recommended = set(
        recommended_movie_ids[:k]
    )

    relevant = set(relevant_movie_ids)

    if len(relevant) == 0:
        return np.nan

    hits = len(
        recommended & relevant
    )

    return hits / len(relevant)

In [13]:
def ndcg_at_k(
    recommended_movie_ids,
    relevant_movie_ids,
    k=10
):
    recommended = recommended_movie_ids[:k]
    relevant = set(relevant_movie_ids)

    if len(relevant) == 0:
        return np.nan

    # DCG for the actual recommendation ranking
    dcg = 0.0

    for i, movie_id in enumerate(recommended):
        if movie_id in relevant:
            dcg += 1 / np.log2(i + 2)

    # DCG for the ideal recommendation ranking
    ideal_hits = min(len(relevant), k)

    idcg = sum(
        1 / np.log2(i + 2)
        for i in range(ideal_hits)
    )

    return dcg / idcg

In [14]:
recommended_movies = [
    1, 2, 3, 4, 5
]

relevant_movies = [
    1, 3, 6, 7
]

k = 5

In [15]:
precision = precision_at_k(
    recommended_movies,
    relevant_movies,
    k=k
)

recall = recall_at_k(
    recommended_movies,
    relevant_movies,
    k=k
)

ndcg = ndcg_at_k(
    recommended_movies,
    relevant_movies,
    k=k
)

print("=== Evaluation Metric Validation ===")

print("Precision@5:", precision)
print("Recall@5:", recall)
print("NDCG@5:", ndcg)

=== Evaluation Metric Validation ===
Precision@5: 0.4
Recall@5: 0.5
NDCG@5: 0.5855700749881525


In [16]:
user_id = 1

user_test = test_ratings[
    test_ratings["userId"] == user_id
]

relevant_test_movies = user_test[
    user_test["rating"] >= 4.0
]["movieId"].tolist()

print("=== User 1 Test Data ===")

print("Number of test ratings:", len(user_test))

print(
    "Number of relevant movies:",
    len(relevant_test_movies)
)

print("Relevant movie IDs:", relevant_test_movies)

=== User 1 Test Data ===
Number of test ratings: 47
Number of relevant movies: 39
Relevant movie IDs: [2959, 1348, 2644, 2654, 1278, 2459, 2648, 2716, 1092, 3386, 1625, 3273, 1265, 943, 1226, 163, 2291, 1270, 1240, 1206, 3702, 3033, 593, 47, 2353, 3147, 527, 5060, 1090, 1224, 151, 3448, 1298, 3053, 157, 553, 2478, 2012, 2492]


In [17]:
user_id = 1

user_train = train_ratings[
    train_ratings["userId"] == user_id
].copy()

print("=== User 1 Training Data ===")
print("Number of training ratings:", len(user_train))
print(user_train.head(10))

=== User 1 Training Data ===
Number of training ratings: 185
     userId  movieId  rating  timestamp  rating_order  user_rating_count
43        1      804     4.0  964980499             0                232
73        1     1210     5.0  964980499             1                232
120       1     2018     5.0  964980523             2                232
171       1     2628     4.0  964980523             3                232
183       1     2826     4.0  964980523             4                232
219       1     3578     5.0  964980668             5                232
220       1     3617     4.0  964980683             6                232
227       1     3744     4.0  964980694             7                232
6         1      101     5.0  964980868             8                232
24        1      441     4.0  964980868             9                232


In [18]:
# 1. Merge user ratings with movie genre information
user_movies = user_train.merge(
    movie_features_train[
        ["movieId"] + available_genres
    ],
    on="movieId",
    how="left",
    validate="many_to_one"
)

# 2. Initialize genre preferences
user_profile = pd.Series(
    0.0,
    index=available_genres
)

# 3. Incorporate training ratings
for _, row in user_movies.iterrows():

    genre_vector = row[available_genres].astype(float)

    genre_count = genre_vector.sum()

    if genre_count == 0:
        continue

    preference_weight = row["rating"] - 3.5

    user_profile += (
        genre_vector * preference_weight / genre_count
    )

print("=== User 1 Genre Profile ===")

print(
    user_profile.sort_values(
        ascending=False
    )
)

=== User 1 Genre Profile ===
Comedy         23.647619
Adventure      23.155952
Action         20.250000
Drama          18.441667
Children       11.814286
Crime          11.397619
Animation       9.147619
Thriller        9.066667
Fantasy         8.797619
Musical         6.858333
Sci-Fi          6.441667
Romance         4.091667
War             4.041667
Mystery         2.680952
Western         0.791667
Film-Noir       0.375000
Documentary     0.000000
Horror         -0.500000
dtype: float64


In [19]:
from sklearn.metrics.pairwise import cosine_similarity

# 1. Select only movies with genre information
candidates = movie_features_train[
    movie_features_train[available_genres].sum(axis=1) > 0
].copy()

# 2. Prepare user and movie vectors
user_vector = user_profile.to_numpy().reshape(1, -1)

movie_vectors = candidates[
    available_genres
].to_numpy()

# 3. Calculate genre similarity
candidates["genre_similarity"] = cosine_similarity(
    user_vector,
    movie_vectors
)[0]

# 4. Calculate content-based recommendation scores
alpha = 0.7

candidates["content_score"] = (
    alpha * candidates["genre_similarity"]
    + (1 - alpha) * candidates["normalized_rating"]
)

# 5. Exclude movies rated in the training data
watched_ids = set(user_train["movieId"])

candidates = candidates[
    ~candidates["movieId"].isin(watched_ids)
]

# 6. Use only positive genre similarities
candidates = candidates[
    candidates["genre_similarity"] > 0
]

# 7. Recommend the top 10 movies
user1_top10 = candidates.sort_values(
    by="content_score",
    ascending=False
).head(10)

print("=== User 1 Content-Based Top 10 ===")

print(
    user1_top10[
        [
            "movieId",
            "title",
            "genre_similarity",
            "content_score"
        ]
    ].to_string(index=False)
)

=== User 1 Content-Based Top 10 ===
 movieId                                       title  genre_similarity  content_score
    5657                            Flashback (1990)          0.868698       0.802919
    6990              The Great Train Robbery (1978)          0.868698       0.790702
  117646       Dragonheart 2: A New Beginning (2000)          0.845935       0.786985
  119145         Kingsman: The Secret Service (2015)          0.786379       0.777923
    6016         City of God (Cidade de Deus) (2002)          0.737972       0.772985
  108932                       The Lego Movie (2014)          0.792357       0.770730
    8961                     Incredibles, The (2004)          0.789108       0.770421
    2019 Seven Samurai (Shichinin no samurai) (1954)          0.715854       0.765158
    2890                          Three Kings (1999)          0.802748       0.764764
    1884       Fear and Loathing in Las Vegas (1998)          0.755180       0.764609


In [20]:
recommended_ids = user1_top10["movieId"].tolist()

relevant_ids = user_test[
    user_test["rating"] >= 4.0
]["movieId"].tolist()

precision_10 = precision_at_k(
    recommended_ids,
    relevant_ids,
    k=10
)

recall_10 = recall_at_k(
    recommended_ids,
    relevant_ids,
    k=10
)

ndcg_10 = ndcg_at_k(
    recommended_ids,
    relevant_ids,
    k=10
)

print("=== User 1 Evaluation Results ===")

print("Precision@10:", precision_10)
print("Recall@10:", recall_10)
print("NDCG@10:", ndcg_10)

=== User 1 Evaluation Results ===
Precision@10: 0.0
Recall@10: 0.0
NDCG@10: 0.0


In [21]:
evaluation_detail = user1_top10[
    ["movieId", "title", "content_score"]
].merge(
    user_test[["movieId", "rating"]],
    on="movieId",
    how="left"
)

evaluation_detail["relevant"] = (
    evaluation_detail["rating"] >= 4.0
)

print("=== User 1 Recommendation Evaluation Detail ===")

print(evaluation_detail.to_string(index=False))

=== User 1 Recommendation Evaluation Detail ===
 movieId                                       title  content_score  rating  relevant
    5657                            Flashback (1990)       0.802919     NaN     False
    6990              The Great Train Robbery (1978)       0.790702     NaN     False
  117646       Dragonheart 2: A New Beginning (2000)       0.786985     NaN     False
  119145         Kingsman: The Secret Service (2015)       0.777923     NaN     False
    6016         City of God (Cidade de Deus) (2002)       0.772985     NaN     False
  108932                       The Lego Movie (2014)       0.770730     NaN     False
    8961                     Incredibles, The (2004)       0.770421     NaN     False
    2019 Seven Samurai (Shichinin no samurai) (1954)       0.765158     NaN     False
    2890                          Three Kings (1999)       0.764764     NaN     False
    1884       Fear and Loathing in Las Vegas (1998)       0.764609     NaN     False


In [22]:
train_with_genres = train_ratings.merge(
    movie_features_train[
        ["movieId"] + available_genres
    ],
    on="movieId",
    how="left",
    validate="many_to_one"
)

# Count valid genres per movie
genre_counts = train_with_genres[
    available_genres
].sum(axis=1)

# Calculate rating-based preference weights
preference_weights = (
    train_with_genres["rating"] - 3.5
)

# Assign zero contribution to movies without valid genres
genre_contributions = (
    train_with_genres[available_genres]
    .div(genre_counts.replace(0, np.nan), axis=0)
    .mul(preference_weights, axis=0)
    .fillna(0.0)
)

genre_contributions["userId"] = (
    train_with_genres["userId"].values
)

# Accumulate preferences by user
user_profiles = genre_contributions.groupby(
    "userId"
)[available_genres].sum()

print("=== All User Profiles ===")
print("Shape:", user_profiles.shape)
print(user_profiles.head())

=== All User Profiles ===
Shape: (610, 18)
           Action  Adventure  Animation   Children     Comedy      Crime  \
userId                                                                     
1       20.250000  23.155952   9.147619  11.814286  23.647619  11.397619   
2        1.166667   0.166667   0.000000   0.000000   2.833333   1.000000   
3        2.333333  -0.166667  -2.750000  -2.750000  -2.750000  -0.750000   
4       -2.433333  -0.821429   0.803571   0.470238   7.536905   4.208333   
5       -1.175000  -0.575000   0.700000   1.200000  -0.766667   0.291667   

        Documentary      Drama   Fantasy  Film-Noir    Horror   Musical  \
userId                                                                    
1              0.00  18.441667  8.797619   0.375000 -0.500000  6.858333   
2              1.25   1.500000  0.000000   0.000000 -0.166667  0.000000   
3              0.00 -14.000000  0.083333   0.000000  3.750000 -0.750000   
4              1.00   8.311905 -0.091667   0.6666

In [23]:
print("=== User Profile Validation ===")

print(
    np.allclose(
        user_profiles.loc[1].to_numpy(),
        user_profile.to_numpy()
    )
)

print(
    "Maximum difference:",
    np.max(
        np.abs(
            user_profiles.loc[1].to_numpy()
            - user_profile.to_numpy()
        )
    )
)

=== User Profile Validation ===
True
Maximum difference: 1.0658141036401503e-14


In [24]:
evaluation_candidates = movie_features_train[
    movie_features_train[available_genres].sum(axis=1) > 0
].copy()

movie_vectors = evaluation_candidates[
    available_genres
].to_numpy()

movie_ids = evaluation_candidates[
    "movieId"
].to_numpy()

normalized_ratings = evaluation_candidates[
    "normalized_rating"
].to_numpy()

In [25]:
def evaluate_content_model(
    user_id,
    user_profiles,
    train_ratings,
    test_ratings,
    movie_vectors,
    movie_ids,
    normalized_ratings,
    alpha=0.7,
    k=10
):
    # 1. Build the user profile
    user_vector = user_profiles.loc[
        user_id
    ].to_numpy().reshape(1, -1)

    # 2. Handle user profiles without preference signals
    if np.linalg.norm(user_vector) == 0:
        return None

    # 3. Calculate genre similarity
    similarities = cosine_similarity(
        user_vector,
        movie_vectors
    )[0]

    # 4. Calculate content-based scores
    scores = (
        alpha * similarities
        + (1 - alpha) * normalized_ratings
    )

    # 5. Exclude movies rated in the training data
    watched_ids = set(
        train_ratings.loc[
            train_ratings["userId"] == user_id,
            "movieId"
        ]
    )

    # 6. Keep only valid recommendation candidates
    valid_mask = (
        (similarities > 0)
        & ~np.isin(movie_ids, list(watched_ids))
    )

    scores = np.where(
        valid_mask,
        scores,
        -np.inf
    )

    # 7. Recommend the top K movies
    ranking = np.argsort(scores)[::-1]

    ranking = ranking[
        np.isfinite(scores[ranking])
    ][:k]

    recommended_ids = movie_ids[
        ranking
    ].tolist()

    # 8. Extract relevant movies from test data
    relevant_ids = test_ratings.loc[
        (test_ratings["userId"] == user_id)
        & (test_ratings["rating"] >= 4.0),
        "movieId"
    ].tolist()

    # 9. Exclude users with no relevant movies from evaluation
    if len(relevant_ids) == 0:
        return None

    # 10. Calculate evaluation metrics
    return {
        "userId": user_id,
        "precision": precision_at_k(
            recommended_ids, relevant_ids, k
        ),
        "recall": recall_at_k(
            recommended_ids, relevant_ids, k
        ),
        "ndcg": ndcg_at_k(
            recommended_ids, relevant_ids, k
        ),
        "relevant_count": len(relevant_ids),
        "hit_count": len(
            set(recommended_ids) & set(relevant_ids)
        )
    }

In [26]:
evaluation_results = []

for user_id in user_profiles.index:

    result = evaluate_content_model(
        user_id=user_id,
        user_profiles=user_profiles,
        train_ratings=train_ratings,
        test_ratings=test_ratings,
        movie_vectors=movie_vectors,
        movie_ids=movie_ids,
        normalized_ratings=normalized_ratings,
        alpha=0.7,
        k=10
    )

    if result is not None:
        evaluation_results.append(result)

evaluation_df = pd.DataFrame(evaluation_results)

print("=== Content-Based Evaluation ===")

print(
    "Evaluated users:",
    len(evaluation_df)
)

print(
    "Average Precision@10:",
    evaluation_df["precision"].mean()
)

print(
    "Average Recall@10:",
    evaluation_df["recall"].mean()
)

print(
    "Average NDCG@10:",
    evaluation_df["ndcg"].mean()
)

=== Content-Based Evaluation ===
Evaluated users: 592
Average Precision@10: 0.019425675675675675
Average Recall@10: 0.014772050769458357
Average NDCG@10: 0.024358008871364082


In [27]:
print("=== Hit Analysis ===")

print(
    "Users with at least one hit:",
    (evaluation_df["hit_count"] > 0).sum()
)

print(
    "Users with zero hits:",
    (evaluation_df["hit_count"] == 0).sum()
)

print(
    "Hit Rate@10:",
    (evaluation_df["hit_count"] > 0).mean()
)

print("\n=== Metric Distribution ===")
print(
    evaluation_df[
        ["precision", "recall", "ndcg", "hit_count"]
    ].describe()
)

=== Hit Analysis ===
Users with at least one hit: 94
Users with zero hits: 498
Hit Rate@10: 0.15878378378378377

=== Metric Distribution ===
        precision      recall        ndcg   hit_count
count  592.000000  592.000000  592.000000  592.000000
mean     0.019426    0.014772    0.024358    0.194257
std      0.048438    0.046146    0.066569    0.484376
min      0.000000    0.000000    0.000000    0.000000
25%      0.000000    0.000000    0.000000    0.000000
50%      0.000000    0.000000    0.000000    0.000000
75%      0.000000    0.000000    0.000000    0.000000
max      0.300000    0.333333    0.493523    3.000000


In [28]:
def recommend_popular_movies(
    user_id,
    train_ratings,
    movie_features_train,
    k=10
):
    # 1. Identify movies rated in the training data
    watched_ids = set(
        train_ratings.loc[
            train_ratings["userId"] == user_id,
            "movieId"
        ]
    )

    # 2. Identify candidate movies for recommendation
    candidates = movie_features_train[
        ~movie_features_train["movieId"].isin(watched_ids)
    ].copy()

    # 3. Sort by weighted rating
    candidates = candidates.sort_values(
        by="weighted_rating",
        ascending=False
    )

    # 4. Return top-K movies
    return candidates.head(k)["movieId"].tolist()

In [29]:
popular_top10_ids = recommend_popular_movies(
    user_id=1,
    train_ratings=train_ratings,
    movie_features_train=movie_features_train,
    k=10
)

popular_top10 = movie_features_train[
    movie_features_train["movieId"].isin(popular_top10_ids)
].copy()

# Preserve recommendation order
popular_top10["rank"] = (
    popular_top10["movieId"]
    .map({
        movie_id: i + 1
        for i, movie_id in enumerate(popular_top10_ids)
    })
)

popular_top10 = popular_top10.sort_values("rank")

print("=== User 1 Popularity Recommendations ===")

print(
    popular_top10[
        ["rank", "title", "weighted_rating"]
    ].to_string(index=False)
)

=== User 1 Popularity Recommendations ===
 rank                                                                       title  weighted_rating
    1                                            Shawshank Redemption, The (1994)         4.385327
    2                                                       Godfather, The (1972)         4.260163
    3 Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)         4.222991
    4                                                           Fight Club (1999)         4.220147
    5                                                        Departed, The (2006)         4.207527
    6                                                   Lawrence of Arabia (1962)         4.200628
    7                                              Godfather: Part II, The (1974)         4.192115
    8                                                     Dark Knight, The (2008)         4.185726
    9                                                     Schindler

In [30]:
popularity_results = []

for user_id in user_profiles.index:

    recommended_ids = recommend_popular_movies(
        user_id=user_id,
        train_ratings=train_ratings,
        movie_features_train=movie_features_train,
        k=10
    )

    relevant_ids = test_ratings.loc[
        (test_ratings["userId"] == user_id)
        & (test_ratings["rating"] >= 4.0),
        "movieId"
    ].tolist()

    # As in the original evaluation, exclude users without relevant movies
    if len(relevant_ids) == 0:
        continue

    hits = len(
        set(recommended_ids) & set(relevant_ids)
    )

    popularity_results.append({
        "userId": user_id,
        "precision": precision_at_k(
            recommended_ids, relevant_ids, k=10
        ),
        "recall": recall_at_k(
            recommended_ids, relevant_ids, k=10
        ),
        "ndcg": ndcg_at_k(
            recommended_ids, relevant_ids, k=10
        ),
        "hit_count": hits
    })

popularity_df = pd.DataFrame(popularity_results)

print("=== Popularity Baseline Evaluation ===")

print("Evaluated users:", len(popularity_df))
print("Precision@10:", popularity_df["precision"].mean())
print("Recall@10:", popularity_df["recall"].mean())
print("NDCG@10:", popularity_df["ndcg"].mean())
print(
    "Hit Rate@10:",
    (popularity_df["hit_count"] > 0).mean()
)

=== Popularity Baseline Evaluation ===
Evaluated users: 592
Precision@10: 0.04442567567567567
Recall@10: 0.03235768620891032
NDCG@10: 0.055498804117372735
Hit Rate@10: 0.27364864864864863


In [31]:
comparison = pd.DataFrame({
    "Model": [
        "Popularity Baseline",
        "Content-Based"
    ],
    "Precision@10": [
        popularity_df["precision"].mean(),
        evaluation_df["precision"].mean()
    ],
    "Recall@10": [
        popularity_df["recall"].mean(),
        evaluation_df["recall"].mean()
    ],
    "NDCG@10": [
        popularity_df["ndcg"].mean(),
        evaluation_df["ndcg"].mean()
    ],
    "Hit Rate@10": [
        (popularity_df["hit_count"] > 0).mean(),
        (evaluation_df["hit_count"] > 0).mean()
    ]
})

print("=== Model Comparison ===")
print(comparison.to_string(index=False))

=== Model Comparison ===
              Model  Precision@10  Recall@10  NDCG@10  Hit Rate@10
Popularity Baseline      0.044426   0.032358 0.055499     0.273649
      Content-Based      0.019426   0.014772 0.024358     0.158784


In [32]:
train_user_item_matrix = train_ratings.pivot(
    index="userId",
    columns="movieId",
    values="rating"
)

print("=== Training User-Item Matrix ===")

print("Shape:", train_user_item_matrix.shape)

print(
    "Users:",
    train_user_item_matrix.shape[0]
)

print(
    "Movies:",
    train_user_item_matrix.shape[1]
)

print(
    "Observed ratings:",
    train_user_item_matrix.notna().sum().sum()
)

=== Training User-Item Matrix ===
Shape: (610, 8230)
Users: 610
Movies: 8230
Observed ratings: 80419


In [33]:
def calculate_all_item_similarities(
    movie_id,
    user_item_matrix,
    min_common_users=10,
    shrinkage=20
):
    # 1. Get ratings for the target movie
    target_ratings = user_item_matrix[movie_id]

    # 2. Identify users who rated the target movie
    target_users = target_ratings.dropna().index

    # 3. Get all movie ratings from those users
    subset = user_item_matrix.loc[target_users]

    # 4. Count common raters
    common_counts = subset.notna().sum(axis=0)

    # 5. Keep movies with sufficient common raters
    valid_movies = common_counts[
        common_counts >= min_common_users
    ].index

    subset_valid = subset[valid_movies]

    # 6. Check rating variance among common raters for both movies
    target_common = pd.DataFrame(
        {
            col: target_ratings.loc[target_users]
            for col in valid_movies
        }
    ).where(subset_valid.notna())

    valid_variance = (
        (subset_valid.std(axis=0) > 0)
        & (target_common.std(axis=0) > 0)
    )

    subset_valid = subset_valid.loc[
        :, valid_variance
    ]

    # 7. Calculate Pearson correlations
    similarities = subset_valid.corrwith(
        target_ratings.loc[target_users],
        axis=0
    )

    # 8. Apply shrinkage
    adjusted = (
        similarities
        * common_counts.loc[similarities.index]
        / (
            common_counts.loc[similarities.index]
            + shrinkage
        )
    )

    # 9. Restore the full movie index
    results = pd.Series(
        np.nan,
        index=user_item_matrix.columns,
        dtype=float
    )

    results.loc[adjusted.index] = adjusted

    # 10. Exclude the target movie itself
    results.loc[movie_id] = np.nan

    return results

In [34]:
train_similarities = calculate_all_item_similarities(
    movie_id=1,
    user_item_matrix=train_user_item_matrix,
    min_common_users=10,
    shrinkage=20
)

print("=== Train-Based Similarities ===")

print(
    "Number of valid similarities:",
    train_similarities.notna().sum()
)

print(
    "Toy Story vs Jumanji:",
    train_similarities.loc[2]
)

print(
    "Toy Story vs Toy Story 2:",
    train_similarities.loc[3114]
)

=== Train-Based Similarities ===
Number of valid similarities: 1319
Toy Story vs Jumanji: 0.25822784324438747
Toy Story vs Toy Story 2: 0.5383656168776559


In [35]:
train_top_similar = (
    train_similarities
    .dropna()
    .sort_values(ascending=False)
    .head(10)
)

train_top_similar_df = pd.DataFrame({
    "movieId": train_top_similar.index,
    "adjusted_similarity": train_top_similar.values
})

train_top_similar_df = train_top_similar_df.merge(
    movies[["movieId", "title"]],
    on="movieId",
    how="left"
)

print("=== Train-Based Top 10 Similar Movies ===")

print(
    train_top_similar_df.to_string(index=False)
)

=== Train-Based Top 10 Similar Movies ===
 movieId  adjusted_similarity                                      title
    3114             0.538366                         Toy Story 2 (1999)
     588             0.509009                             Aladdin (1992)
    6377             0.506115                        Finding Nemo (2003)
    8961             0.496964                    Incredibles, The (2004)
   78499             0.446734                         Toy Story 3 (2010)
    3408             0.402441                     Erin Brockovich (2000)
     953             0.396151               It's a Wonderful Life (1946)
    2797             0.385682                                 Big (1988)
    2716             0.384471 Ghostbusters (a.k.a. Ghost Busters) (1984)
    4886             0.383738                      Monsters, Inc. (2001)


In [36]:
similarity_cache = {}

def get_cached_similarities(
    movie_id,
    user_item_matrix,
    similarity_cache,
    min_common_users=10,
    shrinkage=20
):
    if movie_id not in similarity_cache:

        similarity_cache[movie_id] = (
            calculate_all_item_similarities(
                movie_id=movie_id,
                user_item_matrix=user_item_matrix,
                min_common_users=min_common_users,
                shrinkage=shrinkage
            )
        )

    return similarity_cache[movie_id]

In [37]:
similarity_cache = {}

similarities_1 = get_cached_similarities(
    movie_id=1,
    user_item_matrix=train_user_item_matrix,
    similarity_cache=similarity_cache
)

print("=== Cache Validation 1 ===")
print("Cached movies:", len(similarity_cache))

similarities_2 = get_cached_similarities(
    movie_id=1,
    user_item_matrix=train_user_item_matrix,
    similarity_cache=similarity_cache
)

print("=== Cache Validation 2 ===")
print("Cached movies:", len(similarity_cache))

print(
    "Results identical:",
    similarities_1.equals(similarities_2)
)

=== Cache Validation 1 ===
Cached movies: 1
=== Cache Validation 2 ===
Cached movies: 1
Results identical: True


In [38]:
def recommend_cf_for_user(
    user_id,
    train_ratings,
    train_user_item_matrix,
    similarity_cache,
    k=10,
    baseline=3.5,
    min_common_users=10,
    shrinkage=20,
    lambda_cf=0.5
):
    # 1. Get the user's training ratings
    user_train = train_ratings[
        train_ratings["userId"] == user_id
    ]

    # 2. Get all candidate movie IDs
    movie_ids = train_user_item_matrix.columns

    numerator = pd.Series(
        0.0,
        index=movie_ids
    )

    denominator = pd.Series(
        0.0,
        index=movie_ids
    )

    positive_support = pd.Series(
        0,
        index=movie_ids
    )

    # 3. Calculate scores for each movie rated by the user
    for row in user_train.itertuples():

        similarities = get_cached_similarities(
            movie_id=row.movieId,
            user_item_matrix=train_user_item_matrix,
            similarity_cache=similarity_cache,
            min_common_users=min_common_users,
            shrinkage=shrinkage
        )

        # Use only positive similarities
        positive_similarities = (
            similarities.clip(lower=0).fillna(0)
        )

        preference = row.rating - baseline

        numerator += positive_similarities * preference
        denominator += positive_similarities

        positive_support += (
            positive_similarities > 0
        ).astype(int)

    # 4. Calculate CF recommendation scores
    cf_scores = numerator / (
        denominator + lambda_cf
    )

    # 5. Exclude already-rated movies
    watched_ids = set(user_train["movieId"])

    # 6. Keep movies with at least two valid similarities
    valid_mask = (
        (positive_support >= 2)
        & (~cf_scores.index.isin(watched_ids))
    )

    cf_scores = cf_scores[valid_mask]

    # 7. Select the top-K highest-scoring movies
    top_movies = cf_scores.sort_values(
        ascending=False
    ).head(k)

    return top_movies.index.tolist()

In [39]:
similarity_cache = {}

user1_cf_ids = recommend_cf_for_user(
    user_id=1,
    train_ratings=train_ratings,
    train_user_item_matrix=train_user_item_matrix,
    similarity_cache=similarity_cache,
    k=10
)

user1_cf_movies = movie_features_train[
    movie_features_train["movieId"].isin(user1_cf_ids)
].copy()

user1_cf_movies["rank"] = (
    user1_cf_movies["movieId"].map({
        movie_id: rank + 1
        for rank, movie_id in enumerate(user1_cf_ids)
    })
)

user1_cf_movies = user1_cf_movies.sort_values("rank")

print("=== User 1 CF Recommendations ===")

print(
    user1_cf_movies[
        ["rank", "movieId", "title"]
    ].to_string(index=False)
)

print(
    "\nNumber of cached movies:",
    len(similarity_cache)
)

=== User 1 CF Recommendations ===
 rank  movieId                                    title
    1    53519                       Death Proof (2007)
    2    30793 Charlie and the Chocolate Factory (2005)
    3    64614                       Gran Torino (2008)
    4    52281                        Grindhouse (2007)
    5     2087                         Peter Pan (1953)
    6     8376                 Napoleon Dynamite (2004)
    7     5995                      Pianist, The (2002)
    8     1235                  Harold and Maude (1971)
    9     5377                       About a Boy (2002)
   10      898           Philadelphia Story, The (1940)

Number of cached movies: 185


In [40]:
user1_relevant_ids = test_ratings.loc[
    (test_ratings["userId"] == 1)
    & (test_ratings["rating"] >= 4.0),
    "movieId"
].tolist()

print("=== User 1 CF Evaluation ===")

print(
    "Precision@10:",
    precision_at_k(
        user1_cf_ids,
        user1_relevant_ids,
        k=10
    )
)

print(
    "Recall@10:",
    recall_at_k(
        user1_cf_ids,
        user1_relevant_ids,
        k=10
    )
)

print(
    "NDCG@10:",
    ndcg_at_k(
        user1_cf_ids,
        user1_relevant_ids,
        k=10
    )
)

=== User 1 CF Evaluation ===
Precision@10: 0.0
Recall@10: 0.0
NDCG@10: 0.0


In [41]:
def evaluate_cf_model(
    user_id,
    train_ratings,
    test_ratings,
    train_user_item_matrix,
    similarity_cache,
    k=10
):
    # 1. Generate CF recommendations for each user
    recommended_ids = recommend_cf_for_user(
        user_id=user_id,
        train_ratings=train_ratings,
        train_user_item_matrix=train_user_item_matrix,
        similarity_cache=similarity_cache,
        k=k
    )

    # 2. Identify relevant movies in test data
    relevant_ids = test_ratings.loc[
        (test_ratings["userId"] == user_id)
        & (test_ratings["rating"] >= 4.0),
        "movieId"
    ].tolist()

    # 3. Exclude users without relevant movies
    if len(relevant_ids) == 0:
        return None

    # 4. Count actual recommendation hits
    hits = len(
        set(recommended_ids) & set(relevant_ids)
    )

    # 5. Return results
    return {
        "userId": user_id,
        "precision": precision_at_k(
            recommended_ids, relevant_ids, k
        ),
        "recall": recall_at_k(
            recommended_ids, relevant_ids, k
        ),
        "ndcg": ndcg_at_k(
            recommended_ids, relevant_ids, k
        ),
        "hit_count": hits,
        "recommendation_count": len(recommended_ids)
    }

In [42]:
# Keep the existing cache.
# Do not reinitialize the cache with similarity_cache = {}.

test_user_ids = evaluation_df[
    "userId"
].head(10).tolist()

cf_test_results = []

for user_id in test_user_ids:

    result = evaluate_cf_model(
        user_id=user_id,
        train_ratings=train_ratings,
        test_ratings=test_ratings,
        train_user_item_matrix=train_user_item_matrix,
        similarity_cache=similarity_cache,
        k=10
    )

    if result is not None:
        cf_test_results.append(result)

cf_test_df = pd.DataFrame(cf_test_results)

print("=== CF Test Evaluation: 10 Users ===")
print(cf_test_df.to_string(index=False))

=== CF Test Evaluation: 10 Users ===
 userId  precision  recall  ndcg  hit_count  recommendation_count
      1        0.0     0.0   0.0          0                    10
      2        0.0     0.0   0.0          0                    10
      4        0.0     0.0   0.0          0                    10
      5        0.0     0.0   0.0          0                    10
      6        0.0     0.0   0.0          0                    10
      7        0.0     0.0   0.0          0                    10
      8        0.0     0.0   0.0          0                    10
      9        0.0     0.0   0.0          0                    10
     10        0.0     0.0   0.0          0                    10
     11        0.0     0.0   0.0          0                    10


In [43]:
print("=== CF Test Average Metrics ===")

print(
    "Evaluated users:",
    len(cf_test_df)
)

print(
    "Average Precision@10:",
    cf_test_df["precision"].mean()
)

print(
    "Average Recall@10:",
    cf_test_df["recall"].mean()
)

print(
    "Average NDCG@10:",
    cf_test_df["ndcg"].mean()
)

print(
    "Hit Rate@10:",
    (cf_test_df["hit_count"] > 0).mean()
)

=== CF Test Average Metrics ===
Evaluated users: 10
Average Precision@10: 0.0
Average Recall@10: 0.0
Average NDCG@10: 0.0
Hit Rate@10: 0.0


In [44]:
print("=== CF Test Diagnostics ===")

print(
    "Number of cached movies:",
    len(similarity_cache)
)

print(
    "Minimum recommendations:",
    cf_test_df["recommendation_count"].min()
)

print(
    "Maximum recommendations:",
    cf_test_df["recommendation_count"].max()
)

print(
    "Users with fewer than 10 recommendations:",
    (
        cf_test_df["recommendation_count"] < 10
    ).sum()
)

=== CF Test Diagnostics ===
Number of cached movies: 769
Minimum recommendations: 10
Maximum recommendations: 10
Users with fewer than 10 recommendations: 0


In [45]:
train_movie_ids = set(
    train_user_item_matrix.columns
)

diagnostic_results = []

for user_id in test_user_ids:

    relevant_ids = set(
        test_ratings.loc[
            (test_ratings["userId"] == user_id)
            & (test_ratings["rating"] >= 4.0),
            "movieId"
        ]
    )

    available_ids = relevant_ids & train_movie_ids

    diagnostic_results.append({
        "userId": user_id,
        "relevant_count": len(relevant_ids),
        "cf_candidate_count": len(available_ids),
        "candidate_coverage": (
            len(available_ids) / len(relevant_ids)
            if len(relevant_ids) > 0
            else np.nan
        )
    })

coverage_df = pd.DataFrame(diagnostic_results)

print("=== CF Candidate Coverage ===")
print(coverage_df.to_string(index=False))

print(
    "\nAverage Candidate Coverage:",
    coverage_df["candidate_coverage"].mean()
)

=== CF Candidate Coverage ===
 userId  relevant_count  cf_candidate_count  candidate_coverage
      1              39                  39            1.000000
      2               5                   5            1.000000
      4              22                  21            0.954545
      5               5                   5            1.000000
      6              21                  21            1.000000
      7               7                   7            1.000000
      8               4                   4            1.000000
      9               1                   1            1.000000
     10               7                   6            0.857143
     11              11                  11            1.000000

Average Candidate Coverage: 0.9811688311688312


In [46]:
rank_diagnostic = []

for user_id in test_user_ids:

    recommended_50 = recommend_cf_for_user(
        user_id=user_id,
        train_ratings=train_ratings,
        train_user_item_matrix=train_user_item_matrix,
        similarity_cache=similarity_cache,
        k=50
    )

    relevant_ids = set(
        test_ratings.loc[
            (test_ratings["userId"] == user_id)
            & (test_ratings["rating"] >= 4.0),
            "movieId"
        ]
    )

    hits_10 = len(
        set(recommended_50[:10]) & relevant_ids
    )

    hits_50 = len(
        set(recommended_50[:50]) & relevant_ids
    )

    rank_diagnostic.append({
        "userId": user_id,
        "hits_at_10": hits_10,
        "hits_at_50": hits_50,
        "recommendation_count": len(recommended_50)
    })

rank_df = pd.DataFrame(rank_diagnostic)

print("=== CF Ranking Diagnostics ===")
print(rank_df.to_string(index=False))

print(
    "\nUsers with hits at 50:",
    (rank_df["hits_at_50"] > 0).sum()
)

=== CF Ranking Diagnostics ===
 userId  hits_at_10  hits_at_50  recommendation_count
      1           0           2                    50
      2           0           0                    50
      4           0           0                    50
      5           0           0                    50
      6           0           0                    50
      7           0           0                    50
      8           0           0                    50
      9           0           0                    50
     10           0           0                    50
     11           0           0                    50

Users with hits at 50: 1


In [47]:
comparison_10 = cf_test_df[
    ["userId", "hit_count"]
].rename(
    columns={"hit_count": "cf_hits"}
)

popularity_10 = popularity_df[
    popularity_df["userId"].isin(test_user_ids)
][["userId", "hit_count"]].rename(
    columns={"hit_count": "popularity_hits"}
)

comparison_10 = comparison_10.merge(
    popularity_10,
    on="userId",
    how="left",
    validate="one_to_one"
)

print("=== CF vs Popularity: Same 10 Users ===")
print(comparison_10.to_string(index=False))

print(
    "\nTotal CF Hits:",
    comparison_10["cf_hits"].sum()
)

print(
    "Total Popularity Hits:",
    comparison_10["popularity_hits"].sum()
)

=== CF vs Popularity: Same 10 Users ===
 userId  cf_hits  popularity_hits
      1        0                2
      2        0                0
      4        0                0
      5        0                0
      6        0                0
      7        0                0
      8        0                0
      9        0                0
     10        0                0
     11        0                1

Total CF Hits: 0
Total Popularity Hits: 3


In [48]:
user_id = 1

user_train = train_ratings[
    train_ratings["userId"] == user_id
]

movie_ids_cf = train_user_item_matrix.columns

numerator = pd.Series(0.0, index=movie_ids_cf)
denominator = pd.Series(0.0, index=movie_ids_cf)
positive_support = pd.Series(0, index=movie_ids_cf)

for row in user_train.itertuples():

    similarities = get_cached_similarities(
        movie_id=row.movieId,
        user_item_matrix=train_user_item_matrix,
        similarity_cache=similarity_cache,
        min_common_users=10,
        shrinkage=20
    )

    positive_similarities = (
        similarities.clip(lower=0).fillna(0)
    )

    preference = row.rating - 3.5

    numerator += positive_similarities * preference
    denominator += positive_similarities

    positive_support += (
        positive_similarities > 0
    ).astype(int)

cf_scores = numerator / (denominator + 0.5)

cf_detail = pd.DataFrame({
    "movieId": movie_ids_cf,
    "cf_score": cf_scores.values,
    "positive_support": positive_support.values
})

print("=== User 1 CF Score Statistics ===")
print(cf_detail["cf_score"].describe())

=== User 1 CF Score Statistics ===
count    8230.000000
mean        0.146588
std         0.311699
min        -0.299577
25%         0.000000
50%         0.000000
75%         0.000000
max         1.203298
Name: cf_score, dtype: float64


In [49]:
relevant_ids = set(
    test_ratings.loc[
        (test_ratings["userId"] == 1)
        & (test_ratings["rating"] >= 4.0),
        "movieId"
    ]
)

relevant_cf = cf_detail[
    cf_detail["movieId"].isin(relevant_ids)
].copy()

relevant_cf["cf_available"] = (
    relevant_cf["positive_support"] >= 2
)

print("=== User 1 Relevant Movie Coverage ===")

print("Relevant movies:", len(relevant_ids))

print(
    "Relevant movies in CF matrix:",
    len(relevant_cf)
)

print(
    "Relevant movies with valid CF support:",
    relevant_cf["cf_available"].sum()
)

print(
    "Relevant movies without valid CF support:",
    (~relevant_cf["cf_available"]).sum()
)

=== User 1 Relevant Movie Coverage ===
Relevant movies: 39
Relevant movies in CF matrix: 39
Relevant movies with valid CF support: 29
Relevant movies without valid CF support: 10


In [50]:
watched_ids = set(user_train["movieId"])

cf_ranked = cf_detail[
    (cf_detail["positive_support"] >= 2)
    & (~cf_detail["movieId"].isin(watched_ids))
].copy()

cf_ranked = cf_ranked.sort_values(
    by=["cf_score", "movieId"],
    ascending=[False, True]
).reset_index(drop=True)

cf_ranked["cf_rank"] = (
    np.arange(len(cf_ranked)) + 1
)

relevant_ranked = cf_ranked[
    cf_ranked["movieId"].isin(relevant_ids)
].copy()

relevant_ranked = relevant_ranked.merge(
    movies[["movieId", "title"]],
    on="movieId",
    how="left"
)

print("=== User 1 Relevant Movie CF Ranks ===")

print(
    relevant_ranked[
        [
            "cf_rank",
            "title",
            "cf_score",
            "positive_support"
        ]
    ].head(20).to_string(index=False)
)

print("\n=== CF Rank Diagnostics ===")

print("Total eligible CF candidates:", len(cf_ranked))

print(
    "Eligible relevant movies:",
    len(relevant_ranked)
)

print(
    "Relevant movies ranked 1-10:",
    (relevant_ranked["cf_rank"] <= 10).sum()
)

print(
    "Relevant movies ranked 11-50:",
    relevant_ranked["cf_rank"].between(11, 50).sum()
)

print(
    "Relevant movies ranked 51 or lower:",
    (relevant_ranked["cf_rank"] > 50).sum()
)

=== User 1 Relevant Movie CF Ranks ===
 cf_rank                                      title  cf_score  positive_support
      12                                 JFK (1991)  1.051066                40
      45                 Clockwork Orange, A (1971)  1.011457                93
      59 Ghostbusters (a.k.a. Ghost Busters) (1984)  0.999666               122
      73                     Green Mile, The (1999)  0.992459                94
     123                             Platoon (1986)  0.968358                79
     149         Back to the Future Part III (1990)  0.956549               101
     171                       Groundhog Day (1993)  0.947206               117
     189           Silence of the Lambs, The (1991)  0.940571               124
     213                Seven (a.k.a. Se7en) (1995)  0.933723               102
     232                  Back to the Future (1985)  0.928342               128
     233                    Schindler's List (1993)  0.928180                79
 

In [51]:
cf_all_results = []

# Use the same users as in the full content-based evaluation
evaluation_user_ids = evaluation_df["userId"].tolist()

# Reset the previous sample-evaluation cache
similarity_cache = {}

for i, user_id in enumerate(evaluation_user_ids):

    result = evaluate_cf_model(
        user_id=user_id,
        train_ratings=train_ratings,
        test_ratings=test_ratings,
        train_user_item_matrix=train_user_item_matrix,
        similarity_cache=similarity_cache,
        k=10
    )

    if result is not None:
        cf_all_results.append(result)

    # Clear the cache every 50 users to limit memory usage
    if (i + 1) % 50 == 0:
        similarity_cache.clear()

    if (i + 1) % 50 == 0:
        print(
            f"Evaluated {i + 1} users "
            f"out of {len(evaluation_user_ids)}"
        )

cf_all_df = pd.DataFrame(cf_all_results)

Evaluated 50 users out of 592
Evaluated 100 users out of 592
Evaluated 150 users out of 592
Evaluated 200 users out of 592
Evaluated 250 users out of 592
Evaluated 300 users out of 592
Evaluated 350 users out of 592
Evaluated 400 users out of 592
Evaluated 450 users out of 592
Evaluated 500 users out of 592
Evaluated 550 users out of 592


In [52]:
print("=== Full CF Evaluation ===")

print(
    "Evaluated users:",
    len(cf_all_df)
)

print(
    "Precision@10:",
    cf_all_df["precision"].mean()
)

print(
    "Recall@10:",
    cf_all_df["recall"].mean()
)

print(
    "NDCG@10:",
    cf_all_df["ndcg"].mean()
)

print(
    "Hit Rate@10:",
    (cf_all_df["hit_count"] > 0).mean()
)

=== Full CF Evaluation ===
Evaluated users: 592
Precision@10: 0.013851351351351354
Recall@10: 0.010588552646892605
NDCG@10: 0.014794307964619096
Hit Rate@10: 0.10304054054054054


In [53]:
print("=== Full CF Diagnostics ===")

print(
    "Users with fewer than 10 recommendations:",
    (
        cf_all_df["recommendation_count"] < 10
    ).sum()
)

print(
    "Users with at least one hit:",
    (cf_all_df["hit_count"] > 0).sum()
)

print(
    "Maximum hits:",
    cf_all_df["hit_count"].max()
)

print(
    "Average recommendation count:",
    cf_all_df["recommendation_count"].mean()
)

=== Full CF Diagnostics ===
Users with fewer than 10 recommendations: 0
Users with at least one hit: 61
Maximum hits: 5
Average recommendation count: 10.0


In [54]:
model_comparison = pd.DataFrame({
    "Model": [
        "Popularity",
        "Content-Based",
        "Collaborative Filtering"
    ],
    "Precision@10": [
        popularity_df["precision"].mean(),
        evaluation_df["precision"].mean(),
        cf_all_df["precision"].mean()
    ],
    "Recall@10": [
        popularity_df["recall"].mean(),
        evaluation_df["recall"].mean(),
        cf_all_df["recall"].mean()
    ],
    "NDCG@10": [
        popularity_df["ndcg"].mean(),
        evaluation_df["ndcg"].mean(),
        cf_all_df["ndcg"].mean()
    ]
})

print("=== Model Comparison ===")
print(model_comparison.to_string(index=False))

=== Model Comparison ===
                  Model  Precision@10  Recall@10  NDCG@10
             Popularity      0.044426   0.032358 0.055499
          Content-Based      0.019426   0.014772 0.024358
Collaborative Filtering      0.013851   0.010589 0.014794


In [55]:
def calculate_cf_scores_for_user(
    user_id,
    train_ratings,
    train_user_item_matrix,
    similarity_cache,
    baseline=3.5,
    lambda_cf=0.5
):
    user_train = train_ratings[
        train_ratings["userId"] == user_id
    ]

    movie_ids = train_user_item_matrix.columns

    numerator = pd.Series(0.0, index=movie_ids)
    denominator = pd.Series(0.0, index=movie_ids)
    positive_support = pd.Series(0, index=movie_ids)

    for row in user_train.itertuples():

        similarities = get_cached_similarities(
            movie_id=row.movieId,
            user_item_matrix=train_user_item_matrix,
            similarity_cache=similarity_cache,
            min_common_users=10,
            shrinkage=20
        )

        positive_similarities = (
            similarities.clip(lower=0).fillna(0)
        )

        preference = row.rating - baseline

        numerator += positive_similarities * preference
        denominator += positive_similarities

        positive_support += (
            positive_similarities > 0
        ).astype(int)

    cf_scores = numerator / (
        denominator + lambda_cf
    )

    # Use the same conditions as the original CF model
    cf_scores = cf_scores.where(
        positive_support >= 2,
        np.nan
    )

    # Exclude already-rated movies
    watched_ids = set(user_train["movieId"])

    cf_scores.loc[
        cf_scores.index.isin(watched_ids)
    ] = np.nan

    return cf_scores

In [56]:
def recommend_hybrid_for_user(
    user_id,
    user_profiles,
    train_ratings,
    train_user_item_matrix,
    similarity_cache,
    movie_vectors,
    movie_ids,
    normalized_ratings,
    alpha=0.7,
    beta=0.2,
    k=10
):
    # 1. Build the user genre profile
    user_vector = user_profiles.loc[
        user_id
    ].to_numpy().reshape(1, -1)

    if np.linalg.norm(user_vector) == 0:
        return []

    # 2. Calculate content-based scores
    similarities = cosine_similarity(
        user_vector,
        movie_vectors
    )[0]

    content_scores = (
        alpha * similarities
        + (1 - alpha) * normalized_ratings
    )

    # 3. Calculate CF scores
    cf_scores = calculate_cf_scores_for_user(
        user_id=user_id,
        train_ratings=train_ratings,
        train_user_item_matrix=train_user_item_matrix,
        similarity_cache=similarity_cache
    )

    # 4. Align content candidates with CF scores
    cf_values = cf_scores.reindex(movie_ids)

    cf_signal = (
        cf_values.fillna(0).to_numpy() / 1.5
    )

    # 5. Calculate hybrid scores
    hybrid_scores = (
        content_scores + beta * cf_signal
    )

    hybrid_scores = np.clip(
        hybrid_scores, 0, 1
    )

    # 6. Exclude already-rated movies
    watched_ids = set(
        train_ratings.loc[
            train_ratings["userId"] == user_id,
            "movieId"
        ]
    )

    # 7. Use the same candidate criteria as the original content-based model
    valid_mask = (
        (similarities > 0)
        & (~np.isin(movie_ids, list(watched_ids)))
    )

    hybrid_scores = np.where(
        valid_mask,
        hybrid_scores,
        -np.inf
    )

    # 8. Top 10
    ranking = np.argsort(
        hybrid_scores
    )[::-1]

    ranking = ranking[
        np.isfinite(hybrid_scores[ranking])
    ][:k]

    return movie_ids[ranking].tolist()

In [57]:
similarity_cache = {}

user1_hybrid_ids = recommend_hybrid_for_user(
    user_id=1,
    user_profiles=user_profiles,
    train_ratings=train_ratings,
    train_user_item_matrix=train_user_item_matrix,
    similarity_cache=similarity_cache,
    movie_vectors=movie_vectors,
    movie_ids=movie_ids,
    normalized_ratings=normalized_ratings,
    alpha=0.7,
    beta=0.2,
    k=10
)

user1_hybrid_movies = movie_features_train[
    movie_features_train["movieId"].isin(user1_hybrid_ids)
].copy()

user1_hybrid_movies["rank"] = (
    user1_hybrid_movies["movieId"].map({
        movie_id: rank + 1
        for rank, movie_id in enumerate(user1_hybrid_ids)
    })
)

user1_hybrid_movies = user1_hybrid_movies.sort_values(
    by="rank"
)

print("=== User 1 Hybrid Recommendations ===")

print(
    user1_hybrid_movies[
        ["rank", "movieId", "title"]
    ].to_string(index=False)
)

=== User 1 Hybrid Recommendations ===
 rank  movieId                                                 title
    1    46578                           Little Miss Sunshine (2006)
    2     6016                   City of God (Cidade de Deus) (2002)
    3   119145                   Kingsman: The Secret Service (2015)
    4     7153 Lord of the Rings: The Return of the King, The (2003)
    5     2019           Seven Samurai (Shichinin no samurai) (1954)
    6   134853                                     Inside Out (2015)
    7     8961                               Incredibles, The (2004)
    8     2890                                    Three Kings (1999)
    9     1884                 Fear and Loathing in Las Vegas (1998)
   10     3000              Princess Mononoke (Mononoke-hime) (1997)


In [58]:
user1_relevant_ids = test_ratings.loc[
    (test_ratings["userId"] == 1)
    & (test_ratings["rating"] >= 4.0),
    "movieId"
].tolist()

print("=== User 1 Hybrid Evaluation ===")

print(
    "Precision@10:",
    precision_at_k(
        user1_hybrid_ids,
        user1_relevant_ids,
        k=10
    )
)

print(
    "Recall@10:",
    recall_at_k(
        user1_hybrid_ids,
        user1_relevant_ids,
        k=10
    )
)

print(
    "NDCG@10:",
    ndcg_at_k(
        user1_hybrid_ids,
        user1_relevant_ids,
        k=10
    )
)

print(
    "Cached movies:",
    len(similarity_cache)
)

=== User 1 Hybrid Evaluation ===
Precision@10: 0.0
Recall@10: 0.0
NDCG@10: 0.0
Cached movies: 185


In [59]:
def evaluate_hybrid_model(
    user_id,
    user_profiles,
    train_ratings,
    test_ratings,
    train_user_item_matrix,
    similarity_cache,
    movie_vectors,
    movie_ids,
    normalized_ratings,
    alpha=0.7,
    beta=0.2,
    k=10
):
    # 1. Generate hybrid recommendations for each user
    recommended_ids = recommend_hybrid_for_user(
        user_id=user_id,
        user_profiles=user_profiles,
        train_ratings=train_ratings,
        train_user_item_matrix=train_user_item_matrix,
        similarity_cache=similarity_cache,
        movie_vectors=movie_vectors,
        movie_ids=movie_ids,
        normalized_ratings=normalized_ratings,
        alpha=alpha,
        beta=beta,
        k=k
    )

    # 2. Identify relevant movies in test data
    relevant_ids = test_ratings.loc[
        (test_ratings["userId"] == user_id)
        & (test_ratings["rating"] >= 4.0),
        "movieId"
    ].tolist()

    # 3. Exclude users without relevant test movies
    if len(relevant_ids) == 0:
        return None

    # 4. Count recommendation hits
    hits = len(
        set(recommended_ids) & set(relevant_ids)
    )

    # 5. Evaluation results
    return {
        "userId": user_id,
        "precision": precision_at_k(
            recommended_ids, relevant_ids, k
        ),
        "recall": recall_at_k(
            recommended_ids, relevant_ids, k
        ),
        "ndcg": ndcg_at_k(
            recommended_ids, relevant_ids, k
        ),
        "hit_count": hits,
        "recommendation_count": len(recommended_ids)
    }

In [60]:
hybrid_all_results = []

evaluation_user_ids = evaluation_df["userId"].tolist()

# Reset the cache previously used for user 1
similarity_cache = {}

for i, user_id in enumerate(evaluation_user_ids):

    result = evaluate_hybrid_model(
        user_id=user_id,
        user_profiles=user_profiles,
        train_ratings=train_ratings,
        test_ratings=test_ratings,
        train_user_item_matrix=train_user_item_matrix,
        similarity_cache=similarity_cache,
        movie_vectors=movie_vectors,
        movie_ids=movie_ids,
        normalized_ratings=normalized_ratings,
        alpha=0.7,
        beta=0.2,
        k=10
    )

    if result is not None:
        hybrid_all_results.append(result)

    # Manage memory usage
    if (i + 1) % 50 == 0:
        similarity_cache.clear()

        print(
            f"Evaluated {i + 1} users "
            f"out of {len(evaluation_user_ids)}"
        )

hybrid_all_df = pd.DataFrame(hybrid_all_results)

Evaluated 50 users out of 592
Evaluated 100 users out of 592
Evaluated 150 users out of 592
Evaluated 200 users out of 592
Evaluated 250 users out of 592
Evaluated 300 users out of 592
Evaluated 350 users out of 592
Evaluated 400 users out of 592
Evaluated 450 users out of 592
Evaluated 500 users out of 592
Evaluated 550 users out of 592


In [61]:
print("=== Full Hybrid Evaluation ===")

print(
    "Evaluated users:",
    len(hybrid_all_df)
)

print(
    "Precision@10:",
    hybrid_all_df["precision"].mean()
)

print(
    "Recall@10:",
    hybrid_all_df["recall"].mean()
)

print(
    "NDCG@10:",
    hybrid_all_df["ndcg"].mean()
)

print(
    "Hit Rate@10:",
    (hybrid_all_df["hit_count"] > 0).mean()
)

=== Full Hybrid Evaluation ===
Evaluated users: 592
Precision@10: 0.02263513513513514
Recall@10: 0.016615448362822704
NDCG@10: 0.028458280588830792
Hit Rate@10: 0.17736486486486486


In [62]:
print("=== Full Hybrid Diagnostics ===")

print(
    "Users with fewer than 10 recommendations:",
    (
        hybrid_all_df["recommendation_count"] < 10
    ).sum()
)

print(
    "Users with at least one hit:",
    (hybrid_all_df["hit_count"] > 0).sum()
)

print(
    "Maximum hits:",
    hybrid_all_df["hit_count"].max()
)

print(
    "Average recommendation count:",
    hybrid_all_df["recommendation_count"].mean()
)

=== Full Hybrid Diagnostics ===
Users with fewer than 10 recommendations: 22
Users with at least one hit: 105
Maximum hits: 3
Average recommendation count: 9.66722972972973


In [63]:
final_comparison = pd.DataFrame({
    "Model": [
        "Popularity",
        "Content-Based",
        "Collaborative Filtering",
        "Hybrid"
    ],
    "Precision@10": [
        popularity_df["precision"].mean(),
        evaluation_df["precision"].mean(),
        cf_all_df["precision"].mean(),
        hybrid_all_df["precision"].mean()
    ],
    "Recall@10": [
        popularity_df["recall"].mean(),
        evaluation_df["recall"].mean(),
        cf_all_df["recall"].mean(),
        hybrid_all_df["recall"].mean()
    ],
    "NDCG@10": [
        popularity_df["ndcg"].mean(),
        evaluation_df["ndcg"].mean(),
        cf_all_df["ndcg"].mean(),
        hybrid_all_df["ndcg"].mean()
    ],
    "Hit Rate@10": [
        (popularity_df["hit_count"] > 0).mean(),
        (evaluation_df["hit_count"] > 0).mean(),
        (cf_all_df["hit_count"] > 0).mean(),
        (hybrid_all_df["hit_count"] > 0).mean()
    ]
})

print("=== Final Model Comparison ===")
print(final_comparison.to_string(index=False))

=== Final Model Comparison ===
                  Model  Precision@10  Recall@10  NDCG@10  Hit Rate@10
             Popularity      0.044426   0.032358 0.055499     0.273649
          Content-Based      0.019426   0.014772 0.024358     0.158784
Collaborative Filtering      0.013851   0.010589 0.014794     0.103041
                 Hybrid      0.022635   0.016615 0.028458     0.177365


In [64]:
print("=== Hybrid Recommendation Count Distribution ===")

print(
    hybrid_all_df["recommendation_count"]
    .value_counts()
    .sort_index()
)

print("\n=== Users With Fewer Than 10 Recommendations ===")

insufficient_users = hybrid_all_df[
    hybrid_all_df["recommendation_count"] < 10
].copy()

print(
    insufficient_users[
        ["userId", "recommendation_count", "hit_count"]
    ].to_string(index=False)
)

=== Hybrid Recommendation Count Distribution ===
recommendation_count
0      15
3       5
4       2
10    570
Name: count, dtype: int64

=== Users With Fewer Than 10 Recommendations ===
 userId  recommendation_count  hit_count
     19                     0          0
     28                     0          0
     36                     0          0
     50                     0          0
     81                     0          0
    132                     3          0
    153                     3          0
    214                     0          0
    217                     0          0
    230                     0          0
    245                     0          0
    288                     3          0
    298                     3          0
    307                     3          0
    342                     0          0
    361                     4          0
    368                     0          0
    386                     0          0
    508                     0      

In [65]:
profile_diagnostics = pd.DataFrame({
    "userId": user_profiles.index,
    "positive_genres": (
        user_profiles > 0
    ).sum(axis=1).values,
    "negative_genres": (
        user_profiles < 0
    ).sum(axis=1).values,
    "profile_norm": np.linalg.norm(
        user_profiles.to_numpy(),
        axis=1
    )
})

profile_diagnostics = profile_diagnostics.merge(
    hybrid_all_df[
        ["userId", "recommendation_count"]
    ],
    on="userId",
    how="inner"
)

print("=== Insufficient Recommendation Profile Analysis ===")

print(
    profile_diagnostics[
        profile_diagnostics["recommendation_count"] < 10
    ].to_string(index=False)
)

print("\n=== Zero-Norm Profiles ===")
print(
    (profile_diagnostics["profile_norm"] == 0).sum()
)

=== Insufficient Recommendation Profile Analysis ===
 userId  positive_genres  negative_genres  profile_norm  recommendation_count
     19                0               17    209.748913                     0
     28                0               18     83.977907                     0
     36                0               16     15.442820                     0
     50                0               18     63.957703                     0
     81                0               12      6.931230                     0
    132                1               17     50.671735                     3
    153                1               17     47.085080                     3
    214                0               13      4.884798                     0
    217                0               17    113.725317                     0
    230                0               16     25.104294                     0
    245                0               10      5.894913                     0
    288    

In [66]:
user_comparison = evaluation_df[
    ["userId", "ndcg"]
].rename(
    columns={"ndcg": "content_ndcg"}
)

user_comparison = user_comparison.merge(
    hybrid_all_df[
        ["userId", "ndcg"]
    ].rename(
        columns={"ndcg": "hybrid_ndcg"}
    ),
    on="userId",
    how="inner",
    validate="one_to_one"
)

user_comparison["ndcg_difference"] = (
    user_comparison["hybrid_ndcg"]
    - user_comparison["content_ndcg"]
)

print("=== User-Level Hybrid Improvement ===")

print(
    "Hybrid improved users:",
    (user_comparison["ndcg_difference"] > 0).sum()
)

print(
    "Hybrid unchanged users:",
    (user_comparison["ndcg_difference"] == 0).sum()
)

print(
    "Hybrid worsened users:",
    (user_comparison["ndcg_difference"] < 0).sum()
)

print(
    "Mean NDCG difference:",
    user_comparison["ndcg_difference"].mean()
)

=== User-Level Hybrid Improvement ===
Hybrid improved users: 64
Hybrid unchanged users: 483
Hybrid worsened users: 45
Mean NDCG difference: 0.004100271717466708


In [67]:
def recommend_hybrid_with_fallback(
    user_id,
    user_profiles,
    train_ratings,
    train_user_item_matrix,
    similarity_cache,
    movie_vectors,
    movie_ids,
    normalized_ratings,
    movie_features_train,
    alpha=0.7,
    beta=0.2,
    k=10
):
    # 1. Generate original hybrid recommendations
    hybrid_ids = recommend_hybrid_for_user(
        user_id=user_id,
        user_profiles=user_profiles,
        train_ratings=train_ratings,
        train_user_item_matrix=train_user_item_matrix,
        similarity_cache=similarity_cache,
        movie_vectors=movie_vectors,
        movie_ids=movie_ids,
        normalized_ratings=normalized_ratings,
        alpha=alpha,
        beta=beta,
        k=k
    )

    # 2. Return hybrid recommendations if enough are available
    if len(hybrid_ids) >= k:
        return hybrid_ids[:k]

    # 3. Generate popularity recommendations if needed
    # The current MovieLens dataset has enough fallback candidates
    popular_ids = recommend_popular_movies(
        user_id=user_id,
        train_ratings=train_ratings,
        movie_features_train=movie_features_train,
        k=k + len(hybrid_ids)
    )

    # 4. Combine results without duplicates
    final_ids = list(hybrid_ids)
    seen = set(final_ids)

    for movie_id in popular_ids:

        if movie_id not in seen:
            final_ids.append(movie_id)
            seen.add(movie_id)

        if len(final_ids) == k:
            break

    return final_ids

In [68]:
similarity_cache = {}

test_users = [19, 132, 605, 1]

fallback_results = []

for user_id in test_users:

    original_ids = recommend_hybrid_for_user(
        user_id=user_id,
        user_profiles=user_profiles,
        train_ratings=train_ratings,
        train_user_item_matrix=train_user_item_matrix,
        similarity_cache=similarity_cache,
        movie_vectors=movie_vectors,
        movie_ids=movie_ids,
        normalized_ratings=normalized_ratings,
        alpha=0.7,
        beta=0.2,
        k=10
    )

    fallback_ids = recommend_hybrid_with_fallback(
        user_id=user_id,
        user_profiles=user_profiles,
        train_ratings=train_ratings,
        train_user_item_matrix=train_user_item_matrix,
        similarity_cache=similarity_cache,
        movie_vectors=movie_vectors,
        movie_ids=movie_ids,
        normalized_ratings=normalized_ratings,
        movie_features_train=movie_features_train,
        alpha=0.7,
        beta=0.2,
        k=10
    )

    watched_ids = set(
        train_ratings.loc[
            train_ratings["userId"] == user_id,
            "movieId"
        ]
    )

    fallback_results.append({
        "userId": user_id,
        "original_count": len(original_ids),
        "fallback_count": len(fallback_ids),
        "unique_count": len(set(fallback_ids)),
        "watched_recommended": len(
            set(fallback_ids) & watched_ids
        ),
        "original_preserved": (
            fallback_ids[:len(original_ids)] == original_ids
        )
    })

fallback_validation_df = pd.DataFrame(fallback_results)

print("=== Fallback Validation ===")
print(fallback_validation_df.to_string(index=False))

=== Fallback Validation ===
 userId  original_count  fallback_count  unique_count  watched_recommended  original_preserved
     19               0              10            10                    0                True
    132               3              10            10                    0                True
    605               4              10            10                    0                True
      1              10              10            10                    0                True


In [69]:
user19_ids = recommend_hybrid_with_fallback(
    user_id=19,
    user_profiles=user_profiles,
    train_ratings=train_ratings,
    train_user_item_matrix=train_user_item_matrix,
    similarity_cache=similarity_cache,
    movie_vectors=movie_vectors,
    movie_ids=movie_ids,
    normalized_ratings=normalized_ratings,
    movie_features_train=movie_features_train,
    alpha=0.7,
    beta=0.2,
    k=10
)

user19_movies = movie_features_train.set_index(
    "movieId"
).loc[user19_ids].reset_index()

print("=== User 19 Fallback Recommendations ===")

print(
    user19_movies[
        ["movieId", "title", "weighted_rating"]
    ].to_string(index=False)
)

=== User 19 Fallback Recommendations ===
 movieId                                                                       title  weighted_rating
     318                                            Shawshank Redemption, The (1994)         4.385327
     858                                                       Godfather, The (1972)         4.260163
     750 Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)         4.222991
   48516                                                        Departed, The (2006)         4.207527
    1204                                                   Lawrence of Arabia (1962)         4.200628
    1221                                              Godfather: Part II, The (1974)         4.192115
   58559                                                     Dark Knight, The (2008)         4.185726
      50                                                  Usual Suspects, The (1995)         4.180365
    1213                                 

In [70]:
fallback_user_ids = hybrid_all_df.loc[
    hybrid_all_df["recommendation_count"] < 10,
    "userId"
].tolist()

print("=== Fallback Target Users ===")

print("Number of fallback users:", len(fallback_user_ids))
print("User IDs:", fallback_user_ids)

=== Fallback Target Users ===
Number of fallback users: 22
User IDs: [19, 28, 36, 50, 81, 132, 153, 214, 217, 230, 245, 288, 298, 307, 342, 361, 368, 386, 508, 517, 599, 605]


In [71]:
similarity_cache = {}

fallback_evaluation_results = []

for user_id in fallback_user_ids:

    recommended_ids = recommend_hybrid_with_fallback(
        user_id=user_id,
        user_profiles=user_profiles,
        train_ratings=train_ratings,
        train_user_item_matrix=train_user_item_matrix,
        similarity_cache=similarity_cache,
        movie_vectors=movie_vectors,
        movie_ids=movie_ids,
        normalized_ratings=normalized_ratings,
        movie_features_train=movie_features_train,
        alpha=0.7,
        beta=0.2,
        k=10
    )

    relevant_ids = test_ratings.loc[
        (test_ratings["userId"] == user_id)
        & (test_ratings["rating"] >= 4.0),
        "movieId"
    ].tolist()

    hits = len(
        set(recommended_ids) & set(relevant_ids)
    )

    fallback_evaluation_results.append({
        "userId": user_id,
        "precision": precision_at_k(
            recommended_ids, relevant_ids, k=10
        ),
        "recall": recall_at_k(
            recommended_ids, relevant_ids, k=10
        ),
        "ndcg": ndcg_at_k(
            recommended_ids, relevant_ids, k=10
        ),
        "hit_count": hits,
        "recommendation_count": len(recommended_ids)
    })

fallback_eval_df = pd.DataFrame(
    fallback_evaluation_results
)

print("=== Fallback Users Evaluation ===")

print(
    fallback_eval_df.to_string(index=False)
)

print(
    "\nFallback users with 10 recommendations:",
    (fallback_eval_df["recommendation_count"] == 10).sum()
)

=== Fallback Users Evaluation ===
 userId  precision   recall     ndcg  hit_count  recommendation_count
     19        0.0 0.000000 0.000000          0                    10
     28        0.1 0.083333 0.078398          1                    10
     36        0.1 0.500000 0.613147          1                    10
     50        0.0 0.000000 0.000000          0                    10
     81        0.0 0.000000 0.000000          0                    10
    132        0.2 0.111111 0.173187          2                    10
    153        0.0 0.000000 0.000000          0                    10
    214        0.0 0.000000 0.000000          0                    10
    217        0.0 0.000000 0.000000          0                    10
    230        0.0 0.000000 0.000000          0                    10
    245        0.0 0.000000 0.000000          0                    10
    288        0.3 0.040541 0.211450          3                    10
    298        0.0 0.000000 0.000000          0         

In [72]:
hybrid_fallback_df = hybrid_all_df.copy()

hybrid_fallback_df = hybrid_fallback_df.set_index(
    "userId"
)

fallback_updates = fallback_eval_df.set_index(
    "userId"
)

hybrid_fallback_df.update(
    fallback_updates
)

hybrid_fallback_df = hybrid_fallback_df.reset_index()

print("=== Hybrid Fallback Validation ===")

print(
    "Evaluated users:",
    len(hybrid_fallback_df)
)

print(
    "Users with fewer than 10 recommendations:",
    (
        hybrid_fallback_df["recommendation_count"] < 10
    ).sum()
)

print(
    "Average recommendation count:",
    hybrid_fallback_df["recommendation_count"].mean()
)

print(
    "Total hits:",
    hybrid_fallback_df["hit_count"].sum()
)

=== Hybrid Fallback Validation ===
Evaluated users: 592
Users with fewer than 10 recommendations: 0
Average recommendation count: 10.0
Total hits: 144


In [73]:
fallback_comparison = pd.DataFrame({
    "Model": [
        "Original Hybrid",
        "Hybrid + Fallback"
    ],
    "Precision@10": [
        hybrid_all_df["precision"].mean(),
        hybrid_fallback_df["precision"].mean()
    ],
    "Recall@10": [
        hybrid_all_df["recall"].mean(),
        hybrid_fallback_df["recall"].mean()
    ],
    "NDCG@10": [
        hybrid_all_df["ndcg"].mean(),
        hybrid_fallback_df["ndcg"].mean()
    ],
    "Hit Rate@10": [
        (hybrid_all_df["hit_count"] > 0).mean(),
        (hybrid_fallback_df["hit_count"] > 0).mean()
    ]
})

print("=== Hybrid Fallback Performance Comparison ===")

print(
    fallback_comparison.to_string(index=False)
)

=== Hybrid Fallback Performance Comparison ===
            Model  Precision@10  Recall@10  NDCG@10  Hit Rate@10
  Original Hybrid      0.022635   0.016615 0.028458     0.177365
Hybrid + Fallback      0.024324   0.017913 0.030676     0.185811


In [74]:
original_coverage = (
    hybrid_all_df["recommendation_count"] == 10
).mean()

fallback_coverage = (
    hybrid_fallback_df["recommendation_count"] == 10
).mean()

print("=== Recommendation Coverage Comparison ===")

print(
    "Original Hybrid Coverage@10:",
    original_coverage
)

print(
    "Hybrid + Fallback Coverage@10:",
    fallback_coverage
)

print(
    "Coverage improvement (percentage points):",
    (fallback_coverage - original_coverage) * 100
)

=== Recommendation Coverage Comparison ===
Original Hybrid Coverage@10: 0.9628378378378378
Hybrid + Fallback Coverage@10: 1.0
Coverage improvement (percentage points): 3.716216216216217


In [75]:
# Use only the original training data
inner_data = train_ratings[
    ["userId", "movieId", "rating", "timestamp"]
].copy()

# Sort ratings chronologically for each user
inner_data = inner_data.sort_values(
    ["userId", "timestamp", "movieId"]
).copy()

# Assign rating order for each user
inner_data["rating_order"] = (
    inner_data.groupby("userId").cumcount()
)

# Count ratings for each user
inner_data["user_rating_count"] = (
    inner_data.groupby("userId")["movieId"]
    .transform("count")
)

# Use the most recent 20% for validation
validation_count = np.maximum(
    1,
    np.ceil(
        inner_data["user_rating_count"] * 0.2
    ).astype(int)
)

inner_train_mask = (
    inner_data["rating_order"]
    < inner_data["user_rating_count"] - validation_count
)

inner_train_ratings = inner_data[
    inner_train_mask
].copy()

validation_ratings = inner_data[
    ~inner_train_mask
].copy()

print("=== Inner Train-Validation Split ===")

print("Original train ratings:", len(train_ratings))
print("Inner train ratings:", len(inner_train_ratings))
print("Validation ratings:", len(validation_ratings))

print(
    "Inner train users:",
    inner_train_ratings["userId"].nunique()
)

print(
    "Validation users:",
    validation_ratings["userId"].nunique()
)

=== Inner Train-Validation Split ===
Original train ratings: 80419
Inner train ratings: 64081
Validation ratings: 16338
Inner train users: 610
Validation users: 610


In [76]:
print("=== Validation Split Verification ===")

print(
    "Total ratings:",
    len(inner_train_ratings) + len(validation_ratings)
)

print(
    "Original train ratings:",
    len(train_ratings)
)

overlap = pd.merge(
    inner_train_ratings[["userId", "movieId"]],
    validation_ratings[["userId", "movieId"]],
    on=["userId", "movieId"]
)

print("Overlap:", len(overlap))

print(
    "Minimum inner train ratings:",
    inner_train_ratings.groupby("userId").size().min()
)

print(
    "Minimum validation ratings:",
    validation_ratings.groupby("userId").size().min()
)

print(
    "Users with relevant validation movies:",
    validation_ratings.loc[
        validation_ratings["rating"] >= 4.0,
        "userId"
    ].nunique()
)

=== Validation Split Verification ===
Total ratings: 80419
Original train ratings: 80419
Overlap: 0
Minimum inner train ratings: 12
Minimum validation ratings: 4
Users with relevant validation movies: 593


In [77]:
inner_last_time = (
    inner_train_ratings.groupby("userId")["timestamp"]
    .max()
)

validation_first_time = (
    validation_ratings.groupby("userId")["timestamp"]
    .min()
)

time_check = pd.DataFrame({
    "inner_last": inner_last_time,
    "validation_first": validation_first_time
})

print("=== Temporal Split Validation ===")

print(
    "Users with incorrect time order:",
    (
        time_check["inner_last"]
        > time_check["validation_first"]
    ).sum()
)

print(
    "Users with same boundary timestamp:",
    (
        time_check["inner_last"]
        == time_check["validation_first"]
    ).sum()
)

=== Temporal Split Validation ===
Users with incorrect time order: 0
Users with same boundary timestamp: 132


In [78]:
# 1. Calculate per-movie rating statistics from inner training data
inner_movie_stats = (
    inner_train_ratings
    .groupby("movieId")["rating"]
    .agg(["count", "mean"])
    .reset_index()
    .rename(columns={
        "count": "rating_count",
        "mean": "rating_mean"
    })
)

# 2. Merge movie metadata
inner_movies = movies.merge(
    inner_movie_stats,
    on="movieId",
    how="left",
    validate="one_to_one"
)

# 3. Calculate the global mean from inner training data
inner_global_mean = inner_train_ratings["rating"].mean()

inner_movies["rating_count"] = (
    inner_movies["rating_count"].fillna(0)
)

inner_movies["rating_mean"] = (
    inner_movies["rating_mean"]
    .fillna(inner_global_mean)
)

# 4. Recalculate weighted ratings
m = 10

v = inner_movies["rating_count"]
R = inner_movies["rating_mean"]
C = inner_global_mean

inner_movies["weighted_rating"] = (
    (v / (v + m)) * R
    + (m / (v + m)) * C
)

print("=== Inner Train Movie Statistics ===")
print("Global mean:", inner_global_mean)

print(
    "Movies with training ratings:",
    (inner_movies["rating_count"] > 0).sum()
)

print(
    "Movies without training ratings:",
    (inner_movies["rating_count"] == 0).sum()
)

=== Inner Train Movie Statistics ===
Global mean: 3.529244237761583
Movies with training ratings: 7140
Movies without training ratings: 2602


In [79]:
# 1. Apply min-max normalization
min_wr = inner_movies["weighted_rating"].min()
max_wr = inner_movies["weighted_rating"].max()

inner_movies["normalized_rating"] = (
    (inner_movies["weighted_rating"] - min_wr)
    / (max_wr - min_wr)
)

# 2. One-hot encode genres
inner_genre_matrix = (
    inner_movies["genres"]
    .str.get_dummies(sep="|")
    .reindex(columns=available_genres, fill_value=0)
)

inner_genre_matrix["movieId"] = (
    inner_movies["movieId"].values
)

# 3. Build the final feature dataset
inner_movie_features = inner_movies.merge(
    inner_genre_matrix,
    on="movieId",
    how="left",
    validate="one_to_one"
)

print("=== Inner Train Movie Features ===")

print("Shape:", inner_movie_features.shape)

print(
    "Missing values:",
    inner_movie_features.isnull().sum().sum()
)

print(
    "Duplicate movie IDs:",
    inner_movie_features["movieId"].duplicated().sum()
)

print(
    "Movies with valid genres:",
    (
        inner_movie_features[available_genres]
        .sum(axis=1) > 0
    ).sum()
)

print(
    "Normalized rating range:",
    inner_movie_features["normalized_rating"].min(),
    inner_movie_features["normalized_rating"].max()
)

=== Inner Train Movie Features ===
Shape: (9742, 25)
Missing values: 0
Duplicate movie IDs: 0
Movies with valid genres: 9708
Normalized rating range: 0.0 1.0


In [80]:
inner_comparison = inner_movie_features[
    ["movieId", "title", "weighted_rating"]
].merge(
    movie_features_train[
        ["movieId", "weighted_rating"]
    ],
    on="movieId",
    suffixes=("_inner", "_train"),
    validate="one_to_one"
)

inner_comparison["difference"] = (
    inner_comparison["weighted_rating_inner"]
    - inner_comparison["weighted_rating_train"]
)

print("=== Inner Train vs Train Comparison ===")

print(
    inner_comparison[
        [
            "title",
            "weighted_rating_inner",
            "weighted_rating_train",
            "difference"
        ]
    ].head(10).to_string(index=False)
)

print(
    "\nMovies with changed weighted ratings:",
    (inner_comparison["difference"].abs() > 1e-10).sum()
)

=== Inner Train vs Train Comparison ===
                             title  weighted_rating_inner  weighted_rating_train  difference
                  Toy Story (1995)               3.897211               3.896351    0.000860
                    Jumanji (1995)               3.453763               3.422844    0.030919
           Grumpier Old Men (1995)               3.239832               3.326081   -0.086249
          Waiting to Exhale (1995)               3.128032               3.116438    0.011594
Father of the Bride Part II (1995)               3.128721               3.125105    0.003616
                       Heat (1995)               3.866156               3.882785   -0.016629
                    Sabrina (1995)               3.067193               3.143965   -0.076772
               Tom and Huck (1995)               3.330188               3.133133    0.197055
               Sudden Death (1995)               3.414622               3.324097    0.090525
                  GoldenEye (1

In [81]:
# Use only inner training data
inner_user_means = (
    inner_train_ratings
    .groupby("userId")["rating"]
    .mean()
)

print("=== Inner Train User Mean Statistics ===")
print(inner_user_means.describe())

print("\n=== User Mean Examples ===")
print(inner_user_means.head(10))

print(
    "\nUsers with mean below 3.5:",
    (inner_user_means < 3.5).sum()
)

=== Inner Train User Mean Statistics ===
count    610.000000
mean       3.665025
std        0.497771
min        1.333333
25%        3.366782
50%        3.703835
75%        4.000000
max        5.000000
Name: rating, dtype: float64

=== User Mean Examples ===
userId
1     4.337838
2     3.805556
3     2.979167
4     3.751825
5     3.571429
6     3.605000
7     3.552083
8     3.724138
9     3.464286
10    3.438202
Name: rating, dtype: float64

Users with mean below 3.5: 209


In [82]:
def build_genre_profiles(
    ratings_data,
    movie_features,
    available_genres,
    method="fixed",
    baseline=3.5
):
    # 1. Merge training ratings with genre information
    data = ratings_data.merge(
        movie_features[
            ["movieId"] + available_genres
        ],
        on="movieId",
        how="left",
        validate="many_to_one"
    )

    # 2. Count valid genres per movie
    genre_counts = data[
        available_genres
    ].sum(axis=1)

    # 3. Center ratings
    if method == "fixed":
        centered_ratings = data["rating"] - baseline

    elif method == "user_mean":
        user_means = data.groupby(
            "userId"
        )["rating"].transform("mean")

        centered_ratings = (
            data["rating"] - user_means
        )

    else:
        raise ValueError(
            "method must be 'fixed' or 'user_mean'"
        )

    # 4. Calculate contributions by genre
    contributions = (
        data[available_genres]
        .div(
            genre_counts.replace(0, np.nan),
            axis=0
        )
        .mul(centered_ratings, axis=0)
        .fillna(0.0)
    )

    contributions["userId"] = data["userId"].values

    # 5. Sum genre weights for each user
    profiles = contributions.groupby(
        "userId"
    )[available_genres].sum()

    return profiles

In [83]:
# Original method: fixed baseline of 3.5
inner_profiles_fixed = build_genre_profiles(
    ratings_data=inner_train_ratings,
    movie_features=inner_movie_features,
    available_genres=available_genres,
    method="fixed",
    baseline=3.5
)

# Improvement candidate: user-mean centering
inner_profiles_centered = build_genre_profiles(
    ratings_data=inner_train_ratings,
    movie_features=inner_movie_features,
    available_genres=available_genres,
    method="user_mean"
)

print("=== Profile Shapes ===")

print("Fixed:", inner_profiles_fixed.shape)
print("Centered:", inner_profiles_centered.shape)

=== Profile Shapes ===
Fixed: (610, 18)
Centered: (610, 18)


In [84]:
fixed_positive_counts = (
    inner_profiles_fixed > 0
).sum(axis=1)

centered_positive_counts = (
    inner_profiles_centered > 0
).sum(axis=1)

print("=== Positive Genre Comparison ===")

print(
    "Fixed: users with zero positive genres:",
    (fixed_positive_counts == 0).sum()
)

print(
    "Centered: users with zero positive genres:",
    (centered_positive_counts == 0).sum()
)

print(
    "Fixed: average positive genres:",
    fixed_positive_counts.mean()
)

print(
    "Centered: average positive genres:",
    centered_positive_counts.mean()
)

=== Positive Genre Comparison ===
Fixed: users with zero positive genres: 23
Centered: users with zero positive genres: 2
Fixed: average positive genres: 8.99672131147541
Centered: average positive genres: 7.580327868852459


In [85]:
fixed_norms = np.linalg.norm(
    inner_profiles_fixed.to_numpy(),
    axis=1
)

centered_norms = np.linalg.norm(
    inner_profiles_centered.to_numpy(),
    axis=1
)

print("=== Zero-Norm Profile Comparison ===")

print(
    "Fixed zero-norm users:",
    np.isclose(fixed_norms, 0).sum()
)

print(
    "Centered zero-norm users:",
    np.isclose(centered_norms, 0).sum()
)

=== Zero-Norm Profile Comparison ===
Fixed zero-norm users: 0
Centered zero-norm users: 2


In [86]:
sample_users = [19, 132, 605]

profile_comparison = pd.DataFrame({
    "userId": sample_users,
    "user_mean": [
        inner_user_means.loc[u]
        for u in sample_users
    ],
    "fixed_positive_genres": [
        fixed_positive_counts.loc[u]
        for u in sample_users
    ],
    "centered_positive_genres": [
        centered_positive_counts.loc[u]
        for u in sample_users
    ]
})

print("=== Sample User Profile Comparison ===")
print(profile_comparison.to_string(index=False))

=== Sample User Profile Comparison ===
 userId  user_mean  fixed_positive_genres  centered_positive_genres
     19   2.605791                      0                        12
    132   3.065611                      2                         9
    605   3.364286                      2                         8


In [87]:
from sklearn.metrics.pairwise import cosine_similarity

# Use movie features calculated from inner training data
inner_candidates = inner_movie_features[
    inner_movie_features[available_genres].sum(axis=1) > 0
].copy()

inner_movie_vectors = inner_candidates[
    available_genres
].to_numpy()

inner_movie_ids = inner_candidates[
    "movieId"
].to_numpy()

inner_normalized_ratings = inner_candidates[
    "normalized_rating"
].to_numpy()

print("=== Validation Candidate Data ===")
print("Candidate movies:", len(inner_candidates))
print("Movie vector shape:", inner_movie_vectors.shape)

=== Validation Candidate Data ===
Candidate movies: 9708
Movie vector shape: (9708, 18)


In [88]:
validation_user_ids = sorted(
    validation_ratings.loc[
        validation_ratings["rating"] >= 4.0,
        "userId"
    ].unique()
)

def evaluate_profiles_on_validation(profiles):
    results = []

    for user_id in validation_user_ids:

        result = evaluate_content_model(
            user_id=user_id,
            user_profiles=profiles,
            train_ratings=inner_train_ratings,
            test_ratings=validation_ratings,
            movie_vectors=inner_movie_vectors,
            movie_ids=inner_movie_ids,
            normalized_ratings=inner_normalized_ratings,
            alpha=0.7,
            k=10
        )

        if result is not None:
            results.append(result)

    return pd.DataFrame(results)


fixed_validation_df = evaluate_profiles_on_validation(
    inner_profiles_fixed
)

centered_validation_df = evaluate_profiles_on_validation(
    inner_profiles_centered
)

print("=== Validation User Counts ===")
print("Fixed:", len(fixed_validation_df))
print("Centered:", len(centered_validation_df))

=== Validation User Counts ===
Fixed: 593
Centered: 591


In [89]:
common_users = sorted(
    set(fixed_validation_df["userId"])
    & set(centered_validation_df["userId"])
)

fixed_common = fixed_validation_df[
    fixed_validation_df["userId"].isin(common_users)
].copy()

centered_common = centered_validation_df[
    centered_validation_df["userId"].isin(common_users)
].copy()

profile_validation_comparison = pd.DataFrame({
    "Model": [
        "Fixed Baseline 3.5",
        "User Mean Centering"
    ],
    "Users": [
        len(fixed_common),
        len(centered_common)
    ],
    "Precision@10": [
        fixed_common["precision"].mean(),
        centered_common["precision"].mean()
    ],
    "Recall@10": [
        fixed_common["recall"].mean(),
        centered_common["recall"].mean()
    ],
    "NDCG@10": [
        fixed_common["ndcg"].mean(),
        centered_common["ndcg"].mean()
    ],
    "Hit Rate@10": [
        (fixed_common["hit_count"] > 0).mean(),
        (centered_common["hit_count"] > 0).mean()
    ]
})

print("=== Profile Validation Comparison ===")
print(
    profile_validation_comparison.to_string(index=False)
)

=== Profile Validation Comparison ===
              Model  Users  Precision@10  Recall@10  NDCG@10  Hit Rate@10
 Fixed Baseline 3.5    591      0.017597   0.021835 0.026711     0.143824
User Mean Centering    591      0.016582   0.015160 0.024677     0.126904


In [90]:
profile_user_comparison = fixed_common[
    ["userId", "ndcg"]
].rename(
    columns={"ndcg": "fixed_ndcg"}
).merge(
    centered_common[
        ["userId", "ndcg"]
    ].rename(
        columns={"ndcg": "centered_ndcg"}
    ),
    on="userId",
    validate="one_to_one"
)

profile_user_comparison["difference"] = (
    profile_user_comparison["centered_ndcg"]
    - profile_user_comparison["fixed_ndcg"]
)

print("=== User Mean Centering Improvement ===")

print(
    "Improved users:",
    (profile_user_comparison["difference"] > 0).sum()
)

print(
    "Unchanged users:",
    (profile_user_comparison["difference"] == 0).sum()
)

print(
    "Worsened users:",
    (profile_user_comparison["difference"] < 0).sum()
)

print(
    "Mean NDCG difference:",
    profile_user_comparison["difference"].mean()
)

=== User Mean Centering Improvement ===
Improved users: 46
Unchanged users: 489
Worsened users: 56
Mean NDCG difference: -0.002034299846071629


In [91]:
def recommend_popularity_aware(
    user_id,
    user_profiles,
    train_ratings,
    movie_vectors,
    movie_ids,
    normalized_ratings,
    gamma=0.4,
    k=10
):
    user_vector = user_profiles.loc[
        user_id
    ].to_numpy().reshape(1, -1)

    # Handle zero-vector profiles separately
    if np.linalg.norm(user_vector) == 0:
        return []

    genre_similarities = cosine_similarity(
        user_vector,
        movie_vectors
    )[0]

    # Calculate the original content-based scores
    content_scores = (
        0.7 * genre_similarities
        + 0.3 * normalized_ratings
    )

    # Incorporate popularity information
    scores = (
        (1 - gamma) * content_scores
        + gamma * normalized_ratings
    )

    watched_ids = set(
        train_ratings.loc[
            train_ratings["userId"] == user_id,
            "movieId"
        ]
    )

    valid_mask = (
        (genre_similarities > 0)
        & (~np.isin(movie_ids, list(watched_ids)))
    )

    scores = np.where(
        valid_mask,
        scores,
        -np.inf
    )

    ranking = np.argsort(scores)[::-1]

    ranking = ranking[
        np.isfinite(scores[ranking])
    ][:k]

    return movie_ids[ranking].tolist()

In [92]:
user_id = 1

popularity_aware_ids = recommend_popularity_aware(
    user_id=user_id,
    user_profiles=inner_profiles_fixed,
    train_ratings=inner_train_ratings,
    movie_vectors=inner_movie_vectors,
    movie_ids=inner_movie_ids,
    normalized_ratings=inner_normalized_ratings,
    gamma=0.4,
    k=10
)

recommendation_detail = (
    inner_movie_features
    .set_index("movieId")
    .loc[popularity_aware_ids]
    .reset_index()
)

print("=== User 1 Popularity-Aware Recommendations ===")

print(
    recommendation_detail[
        ["movieId", "title", "normalized_rating"]
    ].to_string(index=False)
)

=== User 1 Popularity-Aware Recommendations ===
 movieId                                                 title  normalized_rating
    2019           Seven Samurai (Shichinin no samurai) (1954)           0.852624
    7153 Lord of the Rings: The Return of the King, The (2003)           0.858413
    6016                   City of God (Cidade de Deus) (2002)           0.837559
    1884                 Fear and Loathing in Las Vegas (1998)           0.787000
    2959                                     Fight Club (1999)           0.933339
    3000              Princess Mononoke (Mononoke-hime) (1997)           0.812555
   48774                                Children of Men (2006)           0.780601
  122904                                       Deadpool (2016)           0.709233
    3030                                        Yojimbo (1961)           0.803453
    1527                             Fifth Element, The (1997)           0.705788


In [93]:
gamma_values = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

gamma_results = []

for gamma in gamma_values:

    user_metrics = []

    for user_id in validation_user_ids:

        recommended_ids = recommend_popularity_aware(
            user_id=user_id,
            user_profiles=inner_profiles_fixed,
            train_ratings=inner_train_ratings,
            movie_vectors=inner_movie_vectors,
            movie_ids=inner_movie_ids,
            normalized_ratings=inner_normalized_ratings,
            gamma=gamma,
            k=10
        )

        relevant_ids = validation_ratings.loc[
            (validation_ratings["userId"] == user_id)
            & (validation_ratings["rating"] >= 4.0),
            "movieId"
        ].tolist()

        user_metrics.append({
            "precision": precision_at_k(
                recommended_ids, relevant_ids, k=10
            ),
            "recall": recall_at_k(
                recommended_ids, relevant_ids, k=10
            ),
            "ndcg": ndcg_at_k(
                recommended_ids, relevant_ids, k=10
            ),
            "recommendation_count": len(recommended_ids)
        })

    metrics_df = pd.DataFrame(user_metrics)

    gamma_results.append({
        "gamma": gamma,
        "precision": metrics_df["precision"].mean(),
        "recall": metrics_df["recall"].mean(),
        "ndcg": metrics_df["ndcg"].mean(),
        "coverage": (
            metrics_df["recommendation_count"] == 10
        ).mean()
    })

gamma_comparison_df = pd.DataFrame(gamma_results)

print("=== Gamma Validation Results ===")

print(
    gamma_comparison_df.to_string(index=False)
)

=== Gamma Validation Results ===
 gamma  precision   recall     ndcg  coverage
   0.0   0.017538 0.021762 0.026621  0.959528
   0.2   0.023946 0.027570 0.032328  0.959528
   0.4   0.029680 0.031271 0.038741  0.959528
   0.6   0.035582 0.037505 0.047891  0.959528
   0.8   0.036931 0.041108 0.050608  0.959528
   1.0   0.034064 0.041818 0.047361  0.959528


In [94]:
inner_user_item_matrix = inner_train_ratings.pivot(
    index="userId",
    columns="movieId",
    values="rating"
)

print("=== Inner Train CF Matrix ===")
print("Shape:", inner_user_item_matrix.shape)
print(
    "Observed ratings:",
    inner_user_item_matrix.notna().sum().sum()
)

=== Inner Train CF Matrix ===
Shape: (610, 7140)
Observed ratings: 64081


In [95]:
def recommend_popularity_aware_hybrid(
    user_id,
    user_profiles,
    train_ratings,
    train_user_item_matrix,
    similarity_cache,
    movie_vectors,
    movie_ids,
    normalized_ratings,
    gamma=0.8,
    beta=0.2,
    k=10
):
    # 1. Build the user genre profile
    user_vector = user_profiles.loc[
        user_id
    ].to_numpy().reshape(1, -1)

    # Handle zero-vector genre profiles
    if np.linalg.norm(user_vector) == 0:
        return []

    # 2. Calculate genre similarity
    similarities = cosine_similarity(
        user_vector,
        movie_vectors
    )[0]

    # 3. Calculate the original content-based scores
    content_scores = (
        0.7 * similarities
        + 0.3 * normalized_ratings
    )

    # 4. Calculate popularity-aware scores
    popularity_aware_scores = (
        (1 - gamma) * content_scores
        + gamma * normalized_ratings
    )

    # 5. Calculate CF scores
    cf_scores = calculate_cf_scores_for_user(
        user_id=user_id,
        train_ratings=train_ratings,
        train_user_item_matrix=train_user_item_matrix,
        similarity_cache=similarity_cache
    )

    cf_signal = (
        cf_scores.reindex(movie_ids)
        .fillna(0)
        .to_numpy() / 1.5
    )

    # 6. Calculate final hybrid scores
    hybrid_scores = np.clip(
        popularity_aware_scores + beta * cf_signal,
        0,
        1
    )

    # 7. Exclude movies rated in the training data
    watched_ids = set(
        train_ratings.loc[
            train_ratings["userId"] == user_id,
            "movieId"
        ]
    )

    # 8. Retain the original positive genre similarity condition
    valid_mask = (
        (similarities > 0)
        & (~np.isin(movie_ids, list(watched_ids)))
    )

    hybrid_scores = np.where(
        valid_mask,
        hybrid_scores,
        -np.inf
    )

    # 9. Recommend the top K movies
    ranking = np.argsort(
        hybrid_scores
    )[::-1]

    ranking = ranking[
        np.isfinite(hybrid_scores[ranking])
    ][:k]

    return movie_ids[ranking].tolist()

In [96]:
similarity_cache = {}

user1_improved_ids = recommend_popularity_aware_hybrid(
    user_id=1,
    user_profiles=inner_profiles_fixed,
    train_ratings=inner_train_ratings,
    train_user_item_matrix=inner_user_item_matrix,
    similarity_cache=similarity_cache,
    movie_vectors=inner_movie_vectors,
    movie_ids=inner_movie_ids,
    normalized_ratings=inner_normalized_ratings,
    gamma=0.8,
    beta=0.2,
    k=10
)

user1_improved_movies = (
    inner_movie_features
    .set_index("movieId")
    .loc[user1_improved_ids]
    .reset_index()
)

print("=== User 1 Improved Hybrid Recommendations ===")

print(
    user1_improved_movies[
        ["movieId", "title", "normalized_rating"]
    ].to_string(index=False)
)

print(
    "\nNumber of cached movies:",
    len(similarity_cache)
)

=== User 1 Improved Hybrid Recommendations ===
 movieId                                                                       title  normalized_rating
     318                                            Shawshank Redemption, The (1994)           1.000000
    2959                                                           Fight Club (1999)           0.933339
     750 Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)           0.928883
     858                                                       Godfather, The (1972)           0.950223
    1221                                              Godfather: Part II, The (1974)           0.927830
    7153                       Lord of the Rings: The Return of the King, The (2003)           0.858413
   58559                                                     Dark Knight, The (2008)           0.893434
     296                                                         Pulp Fiction (1994)           0.867690
    2019         

In [97]:
def evaluate_recommendations(
    recommended_ids,
    relevant_ids,
    k=10
):
    return {
        "precision": precision_at_k(
            recommended_ids, relevant_ids, k
        ),
        "recall": recall_at_k(
            recommended_ids, relevant_ids, k
        ),
        "ndcg": ndcg_at_k(
            recommended_ids, relevant_ids, k
        ),
        "hit_count": len(
            set(recommended_ids[:k]) & set(relevant_ids)
        ),
        "recommendation_count": len(recommended_ids[:k])
    }

In [98]:
validation_sample_users = validation_user_ids[:10]

validation_relevant = {
    user_id: set(group["movieId"])
    for user_id, group in validation_ratings[
        validation_ratings["rating"] >= 4.0
    ].groupby("userId")
}

similarity_cache = {}

validation_sample_results = []

for user_id in validation_sample_users:

    # 1. Original content-based model
    content_result = evaluate_content_model(
        user_id=user_id,
        user_profiles=inner_profiles_fixed,
        train_ratings=inner_train_ratings,
        test_ratings=validation_ratings,
        movie_vectors=inner_movie_vectors,
        movie_ids=inner_movie_ids,
        normalized_ratings=inner_normalized_ratings,
        alpha=0.7,
        k=10
    )

    # 2. Popularity-Aware Content
    popularity_aware_ids = recommend_popularity_aware(
        user_id=user_id,
        user_profiles=inner_profiles_fixed,
        train_ratings=inner_train_ratings,
        movie_vectors=inner_movie_vectors,
        movie_ids=inner_movie_ids,
        normalized_ratings=inner_normalized_ratings,
        gamma=0.8,
        k=10
    )

    # 3. Original hybrid model
    hybrid_ids = recommend_hybrid_for_user(
        user_id=user_id,
        user_profiles=inner_profiles_fixed,
        train_ratings=inner_train_ratings,
        train_user_item_matrix=inner_user_item_matrix,
        similarity_cache=similarity_cache,
        movie_vectors=inner_movie_vectors,
        movie_ids=inner_movie_ids,
        normalized_ratings=inner_normalized_ratings,
        alpha=0.7,
        beta=0.2,
        k=10
    )

    # 4. Improved hybrid model
    improved_ids = recommend_popularity_aware_hybrid(
        user_id=user_id,
        user_profiles=inner_profiles_fixed,
        train_ratings=inner_train_ratings,
        train_user_item_matrix=inner_user_item_matrix,
        similarity_cache=similarity_cache,
        movie_vectors=inner_movie_vectors,
        movie_ids=inner_movie_ids,
        normalized_ratings=inner_normalized_ratings,
        gamma=0.8,
        beta=0.2,
        k=10
    )

    relevant_ids = validation_relevant[user_id]

    recommendations = {
        "Popularity-Aware Content": popularity_aware_ids,
        "Original Hybrid": hybrid_ids,
        "Popularity-Aware Hybrid": improved_ids
    }

    # Use previously calculated metrics for the original content-based model
    if content_result is not None:
        validation_sample_results.append({
            "userId": user_id,
            "model": "Content-Based",
            **{
                key: content_result[key]
                for key in [
                    "precision", "recall", "ndcg", "hit_count"
                ]
            }
        })

    # Evaluate the remaining three models
    for model_name, recommended_ids in recommendations.items():

        metrics = evaluate_recommendations(
            recommended_ids,
            relevant_ids,
            k=10
        )

        validation_sample_results.append({
            "userId": user_id,
            "model": model_name,
            **metrics
        })

sample_validation_df = pd.DataFrame(
    validation_sample_results
)

print("=== Validation Sample Model Comparison ===")

print(
    sample_validation_df.groupby("model")[
        ["precision", "recall", "ndcg", "hit_count"]
    ].mean().to_string()
)

print("\n=== Sample Diagnostics ===")

print("Evaluated users:", len(validation_sample_users))
print("Cached movies:", len(similarity_cache))

=== Validation Sample Model Comparison ===
                          precision    recall      ndcg  hit_count
model                                                             
Content-Based                  0.00  0.000000  0.000000        0.0
Original Hybrid                0.01  0.007143  0.009479        0.1
Popularity-Aware Content       0.01  0.050000  0.017724        0.1
Popularity-Aware Hybrid        0.02  0.057143  0.030347        0.2

=== Sample Diagnostics ===
Evaluated users: 10
Cached movies: 618


In [99]:
# Evaluation results for both hybrid models
hybrid_validation_results = []

# Use a cache dedicated to inner training data
similarity_cache = {}

for i, user_id in enumerate(validation_user_ids):

    # 1. Original Hybrid
    original_ids = recommend_hybrid_for_user(
        user_id=user_id,
        user_profiles=inner_profiles_fixed,
        train_ratings=inner_train_ratings,
        train_user_item_matrix=inner_user_item_matrix,
        similarity_cache=similarity_cache,
        movie_vectors=inner_movie_vectors,
        movie_ids=inner_movie_ids,
        normalized_ratings=inner_normalized_ratings,
        alpha=0.7,
        beta=0.2,
        k=10
    )

    # 2. Popularity-Aware Hybrid
    improved_ids = recommend_popularity_aware_hybrid(
        user_id=user_id,
        user_profiles=inner_profiles_fixed,
        train_ratings=inner_train_ratings,
        train_user_item_matrix=inner_user_item_matrix,
        similarity_cache=similarity_cache,
        movie_vectors=inner_movie_vectors,
        movie_ids=inner_movie_ids,
        normalized_ratings=inner_normalized_ratings,
        gamma=0.8,
        beta=0.2,
        k=10
    )

    # 3. Identify relevant validation movies
    relevant_ids = validation_relevant[user_id]

    # 4. Evaluate both models using the same function
    for model_name, recommended_ids in [
        ("Original Hybrid", original_ids),
        ("Popularity-Aware Hybrid", improved_ids)
    ]:
        metrics = evaluate_recommendations(
            recommended_ids,
            relevant_ids,
            k=10
        )

        hybrid_validation_results.append({
            "userId": user_id,
            "model": model_name,
            **metrics
        })

    # 5. Clear the cache after every 50 users
    if (i + 1) % 50 == 0:
        similarity_cache.clear()

        print(
            f"Evaluated {i + 1} "
            f"out of {len(validation_user_ids)} users"
        )

hybrid_validation_df = pd.DataFrame(
    hybrid_validation_results
)

Evaluated 50 out of 593 users
Evaluated 100 out of 593 users
Evaluated 150 out of 593 users
Evaluated 200 out of 593 users
Evaluated 250 out of 593 users
Evaluated 300 out of 593 users
Evaluated 350 out of 593 users
Evaluated 400 out of 593 users
Evaluated 450 out of 593 users
Evaluated 500 out of 593 users
Evaluated 550 out of 593 users


In [100]:
print("=== Full Hybrid Validation Results ===")

print(
    hybrid_validation_df.groupby("model")[
        [
            "precision",
            "recall",
            "ndcg",
            "hit_count",
            "recommendation_count"
        ]
    ].mean().to_string()
)

print("\n=== Evaluated User Counts ===")

print(
    hybrid_validation_df.groupby("model")[
        "userId"
    ].nunique()
)

=== Full Hybrid Validation Results ===
                         precision    recall      ndcg  hit_count  recommendation_count
model                                                                                  
Original Hybrid           0.024789  0.030316  0.034332   0.247892               9.63575
Popularity-Aware Hybrid   0.039292  0.043095  0.051081   0.392917               9.63575

=== Evaluated User Counts ===
model
Original Hybrid            593
Popularity-Aware Hybrid    593
Name: userId, dtype: int64


In [101]:
# Original content-based model (gamma=0.0)
content_row = gamma_comparison_df.loc[
    np.isclose(gamma_comparison_df["gamma"], 0.0)
].iloc[0]

# Popularity-Aware Content (gamma=0.8)
popularity_content_row = gamma_comparison_df.loc[
    np.isclose(gamma_comparison_df["gamma"], 0.8)
].iloc[0]

hybrid_summary = (
    hybrid_validation_df
    .groupby("model")[
        ["precision", "recall", "ndcg", "recommendation_count"]
    ]
    .mean()
)

validation_comparison = pd.DataFrame([
    {
        "Model": "Content-Based",
        "Precision@10": content_row["precision"],
        "Recall@10": content_row["recall"],
        "NDCG@10": content_row["ndcg"]
    },
    {
        "Model": "Popularity-Aware Content",
        "Precision@10": popularity_content_row["precision"],
        "Recall@10": popularity_content_row["recall"],
        "NDCG@10": popularity_content_row["ndcg"]
    },
    {
        "Model": "Original Hybrid",
        "Precision@10": hybrid_summary.loc[
            "Original Hybrid", "precision"
        ],
        "Recall@10": hybrid_summary.loc[
            "Original Hybrid", "recall"
        ],
        "NDCG@10": hybrid_summary.loc[
            "Original Hybrid", "ndcg"
        ]
    },
    {
        "Model": "Popularity-Aware Hybrid",
        "Precision@10": hybrid_summary.loc[
            "Popularity-Aware Hybrid", "precision"
        ],
        "Recall@10": hybrid_summary.loc[
            "Popularity-Aware Hybrid", "recall"
        ],
        "NDCG@10": hybrid_summary.loc[
            "Popularity-Aware Hybrid", "ndcg"
        ]
    }
])

print("=== Final Validation Model Comparison ===")

print(
    validation_comparison.to_string(index=False)
)

=== Final Validation Model Comparison ===
                   Model  Precision@10  Recall@10  NDCG@10
           Content-Based      0.017538   0.021762 0.026621
Popularity-Aware Content      0.036931   0.041108 0.050608
         Original Hybrid      0.024789   0.030316 0.034332
 Popularity-Aware Hybrid      0.039292   0.043095 0.051081


In [102]:
print("=== Hybrid Validation Coverage ===")

coverage_summary = (
    hybrid_validation_df.groupby("model")
    .agg(
        evaluated_users=("userId", "nunique"),
        average_recommendations=(
            "recommendation_count", "mean"
        ),
        full_list_users=(
            "recommendation_count",
            lambda x: (x == 10).sum()
        ),
        insufficient_users=(
            "recommendation_count",
            lambda x: (x < 10).sum()
        )
    )
)

coverage_summary["coverage_at_10"] = (
    coverage_summary["full_list_users"]
    / coverage_summary["evaluated_users"]
)

print(coverage_summary.to_string())

=== Hybrid Validation Coverage ===
                         evaluated_users  average_recommendations  full_list_users  insufficient_users  coverage_at_10
model                                                                                                                 
Original Hybrid                      593                  9.63575              569                  24        0.959528
Popularity-Aware Hybrid              593                  9.63575              569                  24        0.959528


In [103]:
def recommend_improved_hybrid_with_fallback(
    user_id,
    similarity_cache,
    k=10
):
    # 1. Generate improved hybrid recommendations
    hybrid_ids = recommend_popularity_aware_hybrid(
        user_id=user_id,
        user_profiles=inner_profiles_fixed,
        train_ratings=inner_train_ratings,
        train_user_item_matrix=inner_user_item_matrix,
        similarity_cache=similarity_cache,
        movie_vectors=inner_movie_vectors,
        movie_ids=inner_movie_ids,
        normalized_ratings=inner_normalized_ratings,
        gamma=0.8,
        beta=0.2,
        k=k
    )

    # 2. Return recommendations if enough are available
    if len(hybrid_ids) == k:
        return hybrid_ids

    # 3. Generate popularity recommendations from inner training data
    popular_ids = recommend_popular_movies(
        user_id=user_id,
        train_ratings=inner_train_ratings,
        movie_features_train=inner_movie_features,
        k=k + len(hybrid_ids)
    )

    # 4. Fill missing recommendations without duplicates
    final_ids = list(hybrid_ids)
    seen = set(final_ids)

    for movie_id in popular_ids:
        if movie_id not in seen:
            final_ids.append(movie_id)
            seen.add(movie_id)

        if len(final_ids) == k:
            break

    return final_ids

In [104]:
improved_hybrid_df = hybrid_validation_df[
    hybrid_validation_df["model"] == "Popularity-Aware Hybrid"
].copy()

fallback_user_ids = improved_hybrid_df.loc[
    improved_hybrid_df["recommendation_count"] < 10,
    "userId"
].tolist()

print("=== Improved Hybrid Fallback Targets ===")
print("Users requiring fallback:", len(fallback_user_ids))

similarity_cache = {}
fallback_results = []

for user_id in fallback_user_ids:

    recommended_ids = recommend_improved_hybrid_with_fallback(
        user_id=user_id,
        similarity_cache=similarity_cache,
        k=10
    )

    relevant_ids = validation_relevant[user_id]

    metrics = evaluate_recommendations(
        recommended_ids,
        relevant_ids,
        k=10
    )

    fallback_results.append({
        "userId": user_id,
        **metrics
    })

fallback_updates_df = pd.DataFrame(fallback_results)

print("\n=== Improved Hybrid Fallback Validation ===")
print("Evaluated users:", len(fallback_updates_df))
print(
    "Users with 10 recommendations:",
    (fallback_updates_df["recommendation_count"] == 10).sum()
)
print(
    "Additional hits:",
    fallback_updates_df["hit_count"].sum()
)

=== Improved Hybrid Fallback Targets ===
Users requiring fallback: 24

=== Improved Hybrid Fallback Validation ===
Evaluated users: 24
Users with 10 recommendations: 24
Additional hits: 10


In [105]:
improved_fallback_df = improved_hybrid_df.set_index(
    "userId"
).copy()

improved_fallback_df.update(
    fallback_updates_df.set_index("userId")
)

improved_fallback_df = improved_fallback_df.reset_index()

print("=== Improved Hybrid Fallback Results ===")

print("Evaluated users:", len(improved_fallback_df))

print(
    "Full-List Coverage@10:",
    (
        improved_fallback_df["recommendation_count"] == 10
    ).mean()
)

print(
    "Precision@10:",
    improved_fallback_df["precision"].mean()
)

print(
    "Recall@10:",
    improved_fallback_df["recall"].mean()
)

print(
    "NDCG@10:",
    improved_fallback_df["ndcg"].mean()
)

print(
    "Hit Rate@10:",
    (improved_fallback_df["hit_count"] > 0).mean()
)

=== Improved Hybrid Fallback Results ===
Evaluated users: 593
Full-List Coverage@10: 1.0
Precision@10: 0.04097807757166948
Recall@10: 0.043963909764556976
NDCG@10: 0.05260013922271049
Hit Rate@10: 0.2596964586846543


In [106]:
popularity_content_ndcg = []

for user_id in validation_user_ids:

    recommended_ids = recommend_popularity_aware(
        user_id=user_id,
        user_profiles=inner_profiles_fixed,
        train_ratings=inner_train_ratings,
        movie_vectors=inner_movie_vectors,
        movie_ids=inner_movie_ids,
        normalized_ratings=inner_normalized_ratings,
        gamma=0.8,
        k=10
    )

    popularity_content_ndcg.append({
        "userId": user_id,
        "content_ndcg": ndcg_at_k(
            recommended_ids,
            validation_relevant[user_id],
            k=10
        )
    })

content_ndcg_df = pd.DataFrame(popularity_content_ndcg)

cf_contribution_df = content_ndcg_df.merge(
    improved_hybrid_df[["userId", "ndcg"]].rename(
        columns={"ndcg": "hybrid_ndcg"}
    ),
    on="userId",
    validate="one_to_one"
)

cf_contribution_df["difference"] = (
    cf_contribution_df["hybrid_ndcg"]
    - cf_contribution_df["content_ndcg"]
)

print("=== CF Contribution Analysis ===")

print(
    "Improved users:",
    (cf_contribution_df["difference"] > 1e-12).sum()
)

print(
    "Unchanged users:",
    (cf_contribution_df["difference"].abs() <= 1e-12).sum()
)

print(
    "Worsened users:",
    (cf_contribution_df["difference"] < -1e-12).sum()
)

print(
    "Mean NDCG difference:",
    cf_contribution_df["difference"].mean()
)

=== CF Contribution Analysis ===
Improved users: 69
Unchanged users: 463
Worsened users: 61
Mean NDCG difference: 0.00047360769747975496


In [107]:
cf_effect = cf_contribution_df.copy()

cf_effect["cf_effect"] = np.select(
    [
        cf_effect["difference"] > 1e-12,
        cf_effect["difference"] < -1e-12
    ],
    [
        "Improved",
        "Worsened"
    ],
    default="Unchanged"
)

print("=== CF Effect Distribution ===")

print(
    cf_effect["cf_effect"].value_counts()
)

=== CF Effect Distribution ===
cf_effect
Unchanged    463
Improved      69
Worsened      61
Name: count, dtype: int64


In [108]:
user_rating_counts = (
    inner_train_ratings
    .groupby("userId")
    .size()
    .rename("train_rating_count")
)

cf_effect = cf_effect.merge(
    user_rating_counts,
    on="userId",
    how="left",
    validate="one_to_one"
)

print("=== CF Effect by Rating Count ===")

print(
    cf_effect.groupby("cf_effect")[
        "train_rating_count"
    ].agg(
        ["count", "mean", "median", "min", "max"]
    ).to_string()
)

=== CF Effect by Rating Count ===
           count        mean  median  min   max
cf_effect                                      
Improved      69  108.768116    60.0   13   713
Unchanged    463  100.507559    44.0   12  1585
Worsened      61  147.704918    61.0   12  1726


In [109]:
user_mean_ratings = (
    inner_train_ratings
    .groupby("userId")["rating"]
    .mean()
    .rename("mean_rating")
)

cf_effect = cf_effect.merge(
    user_mean_ratings,
    on="userId",
    how="left",
    validate="one_to_one"
)

print("=== CF Effect by User Mean Rating ===")

print(
    cf_effect.groupby("cf_effect")[
        "mean_rating"
    ].agg(
        ["count", "mean", "median", "min", "max"]
    ).to_string()
)

=== CF Effect by User Mean Rating ===
           count      mean    median       min       max
cf_effect                                               
Improved      69  3.900770  3.857143  3.300000  4.821429
Unchanged    463  3.633969  3.655844  1.842466  5.000000
Worsened      61  3.843041  3.833333  2.937500  4.645833
